# Null Catamenial Epilepsy Analysis Notebook

Populated from `outputs/random_start_full_v14_waveform_recalibration` outputs. Detected analysis mode: **full**.

- Participants: **100,000** (healthy ovulatory: 50000, heterogeneous menstruating-age: 50000)
- Primary window rows: **2,600,000**
- Study-level Monte Carlo rows: **1,380,000**
- Manifest files: **25**

## Cohort terminology

This notebook uses **heterogeneous menstruating-age** as the presentation label for the broader cohort key stored in the analysis files. In this null-simulation study it means an assumption-driven broader menstruating-age simulated cohort, not a disease-positive, clinically diagnosed, or demographically representative population. This cohort allows the hormone-cycle simulator's natural ovulatory and anovulatory behavior and its configured rates of cycle modifiers such as PCOS, peri-menarche, perimenopause, dysmenorrhea, and cycle irregularity when available. It is contrasted with the **healthy ovulatory** cohort, which is restricted to adult ovulatory cycling with those medical modifiers disabled where the simulator exposes those controls. In both cohorts, seizure and menstrual diaries are generated from separate deterministic random streams. HORMONE-CYCLE selects diary day 1 uniformly from the first generated cycle and then proceeds forward without wrapping. The seizure and menstrual diaries are aligned directly by calendar day without reordering, so any apparent catamenial epilepsy classification is a false positive under the null.

## HORMONE-CYCLE v0.4.0 validation provenance

The primary paper rerun was authorized only after the versioned 10,000-participant adult validation cohort (ages 18–54.9) passed **75/75** calibration/waveform checks and **14/14** held-out Cunningham/Flo checks; **8/8** secondary age-matched modifier software stress tests also passed. The v0.4.0 waveform gate maps every in-cycle Stricker serum observation, uses an independent Anckaert subphase amplitude/order check, and adds Roos-informed timing-dispersion, terminal-P4, anti-time-warp, and cycle-boundary guards. Long follicular phases preserve terminal maturation rather than stretching an ordinary curve. The pass is qualified rather than clinical: the waveform represents a latent daily envelope, not within-day pulsatility or participant-level clinical validation. Cycle-summary agreement is strongest at ages 18–45, and the retained post-50 discrepancy reflects differing variability estimates in AWHS and Flo. Modifier margins are investigator-selected regression guards rather than externally estimated clinical thresholds. The machine-readable report, citation audit, and executable validation notebook are `examples/reports/healthy_cycle_validation_v14.json`, `examples/reports/hormone_citation_audit_v14.json`, and `show_validation.ipynb`.

## Exact analysis plan followed

1. Simulate two cohorts separately and never pool results: healthy ovulatory and heterogeneous menstruating-age.
2. Full defined cohort sizes are `{'healthy ovulatory': 50000, 'heterogeneous menstruating-age': 50000}`; smoke mode uses `100` total participants.
3. For each participant, simulate an independent CHOCOLATES seizure diary and an independent hormone-cycle diary for `36` months in full mode.
4. Select diary day 1 uniformly from the first generated HORMONE-CYCLE cycle, continue forward without wrapping, and align the independently generated seizure and menstrual diaries directly by calendar day.
5. Label phases on the full diary before subsetting windows, using strict Herzog labels for primary analyses and a luteal-anchored fixed ovulatory window for sensitivity analyses.
6. Sample calendar windows, full 36-month windows, and complete-cycle windows exactly as configured.
7. Classify windows using exact Herzog 2004, windowed Herzog thresholds, C3-exclusion and pattern-only sensitivities, minimum-data rules, reproducibility rules, full-window stabilized/window-dispersion NB regression, and assumption-based historical definitions.
8. Summarize false positives and indeterminacy by cohort, phase mode, window, definition, seizure-burden stratum, participant-level status, pattern category, and study-level Monte Carlo benchmarks.
9. Save outputs as parquet/CSV, publication figures as PNG/PDF/SVG, a 1% daily audit sample, and a manifest.

### Recorded assumptions

- Definition D uses a participant-full-diary method-of-moments negative-binomial alpha recorded in d_alpha; Poisson robust fallback is recorded in d_reason when statsmodels NB fitting fails. Definition D_window_alpha re-estimates alpha from the analyzed window as a non-oracle sensitivity.
- HORMONE-CYCLE selected diary day 1 uniformly from the first generated cycle.
- Healthy ovulatory cohort used hormone_cycler build_patient_profile/render_cycle with ovulation_probability set to 1.0 because simulate_diary does not expose a public force-ovulation knob.
- Historical definitions H1-H4 are assumption-based operationalizations and are flagged in summary outputs.
- Large-run non-audit participants used the RNG-equivalent compact hormone path; daily hormone concentrations were omitted, while cycle structure and ILP status were retained.
- Study-level Monte Carlo samples each selected participant from a deterministic pool of precomputed random valid 3-month windows to avoid retaining all daily diaries in memory.
- The hormone simulator exposes medical-factor knobs but no natural prevalence sampler; heterogeneous menstruating-age medical factors were sampled from config.yaml rates.

## Reproducible function calls

These cells are the exact calls used to regenerate the analysis outputs. Run the smoke call for a quick end-to-end check; run the full call for the defined 100,000-participant analysis.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from paper1_null_ce.core.utils import load_config
from paper1_null_ce.core.simulate import run_pipeline

config = load_config(ROOT / "config_random_start_full.yaml")

# Quick validation run used while developing and reviewing the pipeline. It is
# intentionally opt-in so executing this results notebook does not overwrite
# the already-populated definitive artifacts:
# smoke_result = run_pipeline(config, mode="smoke")

# Prespecified full analysis. This is intentionally separate because it is large:
# full_result = run_pipeline(config, mode="full")

# During a long run, check ETA from another terminal:
# python3.11 scripts/check_paper1_progress.py --progress outputs/random_start_full_v14_waveform_recalibration/progress.json


## Load the current populated outputs

The remaining notebook cells read the existing output artifacts. This keeps figure and table rendering fast and reproducible after either a smoke run or a full run.

In [2]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = ROOT / "outputs/random_start_full_v14_waveform_recalibration"

participant_summary = pd.read_parquet(OUTPUT_DIR / "participant_summary.parquet")
window_results = pd.read_parquet(OUTPUT_DIR / "window_results.parquet")
study_path = OUTPUT_DIR / "study_level_3month.parquet"
if not study_path.exists():
    study_path = OUTPUT_DIR / "study_level_3month_n30.parquet"
study_level = pd.read_parquet(study_path)
summary_tables = pd.read_csv(OUTPUT_DIR / "summary_tables.csv")
manifest = json.loads((OUTPUT_DIR / "manifest.json").read_text())

participant_summary.shape, window_results.shape, study_level.shape, summary_tables.shape


/Users/runner/work/crossbow/crossbow/arrow/cpp/src/arrow/util/cpu_info.cc:242: IOError: sysctlbyname failed for 'hw.l1dcachesize'. Detail: [errno 1] Operation not permitted
/Users/runner/work/crossbow/crossbow/arrow/cpp/src/arrow/util/cpu_info.cc:242: IOError: sysctlbyname failed for 'hw.l2cachesize'. Detail: [errno 1] Operation not permitted
/Users/runner/work/crossbow/crossbow/arrow/cpp/src/arrow/util/cpu_info.cc:242: IOError: sysctlbyname failed for 'hw.l3cachesize'. Detail: [errno 1] Operation not permitted
/Users/runner/work/crossbow/crossbow/arrow/cpp/src/arrow/util/cpu_info.cc:242: IOError: sysctlbyname failed for 'hw.optional.neon'. Detail: [errno 1] Operation not permitted


((100000, 28), (2600000, 109), (1380000, 9), (19703, 24))

## Table 1. Cohort and diary summary

**Why this table is included.** This table verifies that both defined cohorts are represented separately, that cycle summaries are available, and that seizure-burden metrics were carried through from the seizure simulator. It is the first QC table because every downstream apparent-classification estimate depends on the cohort construction and diary burden.

**Code to call.**

In [3]:
cohort_summary = (
    participant_summary
    .groupby("cohort")
    .agg(
        participants=("participant_id", "nunique"),
        age_mean=("age", "mean"),
        age_sd=("age", "std"),
        mean_cycle_length=("mean_cycle_length", "mean"),
        sd_cycle_length=("sd_cycle_length", "mean"),
        ovulatory_fraction=("ovulatory_fraction", "mean"),
        seizure_days_per_month=("seizure_days_per_month", "mean"),
        seizures_per_month=("seizures_per_month", "mean"),
    )
    .reset_index()
)
cohort_summary


,cohort,participants,age_mean,age_sd,mean_cycle_length,sd_cycle_length,ovulatory_fraction,seizure_days_per_month,seizures_per_month
0,healthy_ovulatory,50000,31.519764,7.801985,28.753410,3.931374,1.000000,2.458204,6.831809
1,population,50000,33.968544,12.138328,30.092767,5.608094,0.789458,2.454090,6.796666


| Cohort                         | Participants | Mean age, years | Age SD, years | Mean cycle length, days | Mean cycle-length SD, days | Ovulatory cycles | Seizure days per month | Seizures per month |
| ------------------------------ | ------------ | --------------- | ------------- | ----------------------- | -------------------------- | ---------------- | ---------------------- | ------------------ |
| healthy ovulatory              | 50,000       | 31.5            | 7.8           | 28.75                   | 3.93                       | 100.0%           | 2.46                   | 6.83               |
| heterogeneous menstruating-age | 50,000       | 34.0            | 12.1          | 30.09                   | 5.61                       | 78.9%            | 2.45                   | 6.80               |

**Table 1 caption.** Cohort-level participant and diary summaries for the full simulation. Percentages use a 0-100% scale; seizure rates are monthly averages over the 36-month diary.

## Table 2. Primary full-window false-positive rates

**Why this table is included.** This table is the primary result summary for person-window false-positive rates under the null. It uses the full diary window and reports classifiable denominators, positives, Wilson 95% intervals, and indeterminate rates separately by cohort and definition.

**Code to call.**

In [4]:
primary_full = summary_tables[
    (summary_tables.table_type == "window_false_positive")
    & (summary_tables.phase_mode == "strict_herzog")
    & (summary_tables.subset == "all")
    & (summary_tables.window_type == "full")
    & (summary_tables.definition.isin([
        "A_windowed_any", "A_windowed_C1_or_C2",
        "B_minimum_data_any", "B_minimum_data_C1_or_C2",
        "C_reproducibility_any", "D_nb_regression_C1_or_C2"
    ]))
].copy()
primary_full


,table_type,subset,cohort,window_type,window_value,definition,phase_mode,assumption_based_historical,n_windows,n_classifiable,...,indeterminate_rate,positive_rate_all_attempted,unstable_denominator,interpretation_note,n_participants,pattern_category,indeterminate_reason,n_indeterminate,p_prevalence_ge_39_1,p_prevalence_ge_44_2
77,window_false_positive,all,healthy_ovulatory,full,full_diary,A_windowed_any,strict_herzog,False,50000,49605.0,...,0.00790,0.11262,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
103,window_false_positive,all,population,full,full_diary,A_windowed_any,strict_herzog,False,50000,49587.0,...,0.00826,0.35548,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
181,window_false_positive,all,healthy_ovulatory,full,full_diary,A_windowed_C1_or_C2,strict_herzog,False,50000,49605.0,...,0.00790,0.11262,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
207,window_false_positive,all,population,full,full_diary,A_windowed_C1_or_C2,strict_herzog,False,50000,49587.0,...,0.00826,0.11484,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
389,window_false_positive,all,healthy_ovulatory,full,full_diary,B_minimum_data_any,strict_herzog,False,50000,48359.0,...,0.03282,0.09986,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
415,window_false_positive,all,population,full,full_diary,B_minimum_data_any,strict_herzog,False,50000,48333.0,...,0.03334,0.34006,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
493,window_false_positive,all,healthy_ovulatory,full,full_diary,B_minimum_data_C1_or_C2,strict_herzog,False,50000,48359.0,...,0.03282,0.09986,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
519,window_false_positive,all,population,full,full_diary,B_minimum_data_C1_or_C2,strict_herzog,False,50000,48333.0,...,0.03334,0.10140,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
545,window_false_positive,all,healthy_ovulatory,full,full_diary,C_reproducibility_any,strict_herzog,False,50000,5914.0,...,0.88172,0.00000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
571,window_false_positive,all,population,full,full_diary,C_reproducibility_any,strict_herzog,False,50000,12710.0,...,0.74580,0.02950,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


| Cohort                         | CE definition                           | Windows analyzed | Classifiable windows | False-positive windows | False-positive rate (95% CI) | Indeterminate windows |
| ------------------------------ | --------------------------------------- | ---------------- | -------------------- | ---------------------- | ---------------------------- | --------------------- |
| healthy ovulatory              | Windowed Herzog C1/C2 union             | 50,000           | 49,605               | 5,631                  | 11.4% (11.1, 11.6)           | 0.8%                  |
| healthy ovulatory              | Windowed Herzog thresholds              | 50,000           | 49,605               | 5,631                  | 11.4% (11.1, 11.6)           | 0.8%                  |
| healthy ovulatory              | Windowed Herzog C1/C2 with minimum data | 50,000           | 48,359               | 4,993                  | 10.3% (10.1, 10.6)           | 3.3%                  |
| healthy ovulatory              | Windowed Herzog with minimum data       | 50,000           | 48,359               | 4,993                  | 10.3% (10.1, 10.6)           | 3.3%                  |
| healthy ovulatory              | Cycle reproducibility, 6-cycle rule     | 50,000           | 5,914                | 0                      | 0.0% (0.0, 0.1)              | 88.2%                 |
| healthy ovulatory              | Negative-binomial regression C1/C2      | 50,000           | 48,359               | 2,077                  | 4.3% (4.1, 4.5)              | 3.3%                  |
| heterogeneous menstruating-age | Windowed Herzog C1/C2 union             | 50,000           | 49,587               | 5,742                  | 11.6% (11.3, 11.9)           | 0.8%                  |
| heterogeneous menstruating-age | Windowed Herzog thresholds              | 50,000           | 49,587               | 17,774                 | 35.8% (35.4, 36.3)           | 0.8%                  |
| heterogeneous menstruating-age | Windowed Herzog C1/C2 with minimum data | 50,000           | 48,333               | 5,070                  | 10.5% (10.2, 10.8)           | 3.3%                  |
| heterogeneous menstruating-age | Windowed Herzog with minimum data       | 50,000           | 48,333               | 17,003                 | 35.2% (34.8, 35.6)           | 3.3%                  |
| heterogeneous menstruating-age | Cycle reproducibility, 6-cycle rule     | 50,000           | 12,710               | 1,475                  | 11.6% (11.1, 12.2)           | 74.6%                 |
| heterogeneous menstruating-age | Negative-binomial regression C1/C2      | 50,000           | 48,333               | 2,063                  | 4.3% (4.1, 4.5)              | 3.3%                  |

**Table 2 caption.** Primary full-diary false-positive rates under the null. The denominator for the false-positive rate is the number of classifiable participant windows, and the confidence interval is Wilson 95%.

## Table 3. Window-length sensitivity for core definitions

**Why this table is included.** This table shows why diary length matters. Short calendar windows can be classifiable for simple windowed ratios but not for minimum-data, reproducibility, or exact three-cycle rules; the indeterminate column quantifies that tradeoff.

**Code to call.**

In [5]:
window_sensitivity = summary_tables[
    (summary_tables.table_type == "window_false_positive")
    & (summary_tables.phase_mode == "strict_herzog")
    & (summary_tables.subset == "all")
    & (summary_tables.definition.isin([
        "A_exact_any", "A_windowed_any", "A_windowed_C1_or_C2",
        "B_minimum_data_C1_or_C2", "C_reproducibility_C1_or_C2",
        "D_nb_regression_C1_or_C2"
    ]))
].copy()
window_sensitivity


,table_type,subset,cohort,window_type,window_value,definition,phase_mode,assumption_based_historical,n_windows,n_classifiable,...,indeterminate_rate,positive_rate_all_attempted,unstable_denominator,interpretation_note,n_participants,pattern_category,indeterminate_reason,n_indeterminate,p_prevalence_ge_39_1,p_prevalence_ge_44_2
13,window_false_positive,all,healthy_ovulatory,calendar,1,A_exact_any,strict_herzog,False,50000,0.0,...,1.00000,0.00000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14,window_false_positive,all,healthy_ovulatory,calendar,3,A_exact_any,strict_herzog,False,50000,0.0,...,1.00000,0.00000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15,window_false_positive,all,healthy_ovulatory,calendar,4,A_exact_any,strict_herzog,False,50000,0.0,...,1.00000,0.00000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16,window_false_positive,all,healthy_ovulatory,calendar,6,A_exact_any,strict_herzog,False,50000,0.0,...,1.00000,0.00000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,window_false_positive,all,healthy_ovulatory,calendar,9,A_exact_any,strict_herzog,False,50000,0.0,...,1.00000,0.00000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
827,window_false_positive,all,population,calendar,36,D_nb_regression_C1_or_C2,strict_herzog,False,50000,0.0,...,1.00000,0.00000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
828,window_false_positive,all,population,cycle,3,D_nb_regression_C1_or_C2,strict_herzog,False,50000,0.0,...,1.00000,0.00000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
829,window_false_positive,all,population,cycle,6,D_nb_regression_C1_or_C2,strict_herzog,False,50000,0.0,...,1.00000,0.00000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
830,window_false_positive,all,population,cycle,12,D_nb_regression_C1_or_C2,strict_herzog,False,50000,0.0,...,1.00000,0.00000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


| Cohort                         | Observation window  | CE definition                             | Classifiable windows | False-positive windows | False-positive rate | Indeterminate windows |
| ------------------------------ | ------------------- | ----------------------------------------- | -------------------- | ---------------------- | ------------------- | --------------------- |
| healthy ovulatory              | 1 month             | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 1 month             | Windowed Herzog C1/C2 union               | 38,186               | 18,100                 | 47.4%               | 23.6%                 |
| healthy ovulatory              | 1 month             | Windowed Herzog thresholds                | 38,186               | 18,100                 | 47.4%               | 23.6%                 |
| healthy ovulatory              | 1 month             | Windowed Herzog C1/C2 with minimum data   | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 1 month             | Cycle reproducibility C1/C2, 6-cycle rule | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 1 month             | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 12 cycles           | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 12 cycles           | Windowed Herzog C1/C2 union               | 48,668               | 12,380                 | 25.4%               | 2.7%                  |
| healthy ovulatory              | 12 cycles           | Windowed Herzog thresholds                | 48,668               | 12,380                 | 25.4%               | 2.7%                  |
| healthy ovulatory              | 12 cycles           | Windowed Herzog C1/C2 with minimum data   | 45,033               | 10,498                 | 23.3%               | 9.9%                  |
| healthy ovulatory              | 12 cycles           | Cycle reproducibility C1/C2, 6-cycle rule | 15,390               | 307                    | 2.0%                | 69.2%                 |
| healthy ovulatory              | 12 cycles           | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 12 months           | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 12 months           | Windowed Herzog C1/C2 union               | 48,731               | 12,177                 | 25.0%               | 2.5%                  |
| healthy ovulatory              | 12 months           | Windowed Herzog thresholds                | 48,731               | 12,177                 | 25.0%               | 2.5%                  |
| healthy ovulatory              | 12 months           | Windowed Herzog C1/C2 with minimum data   | 45,268               | 10,394                 | 23.0%               | 9.5%                  |
| healthy ovulatory              | 12 months           | Cycle reproducibility C1/C2, 6-cycle rule | 15,667               | 256                    | 1.6%                | 68.7%                 |
| healthy ovulatory              | 12 months           | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 18 months           | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 18 months           | Windowed Herzog C1/C2 union               | 49,159               | 9,564                  | 19.5%               | 1.7%                  |
| healthy ovulatory              | 18 months           | Windowed Herzog thresholds                | 49,159               | 9,564                  | 19.5%               | 1.7%                  |
| healthy ovulatory              | 18 months           | Windowed Herzog C1/C2 with minimum data   | 46,723               | 8,341                  | 17.9%               | 6.6%                  |
| healthy ovulatory              | 18 months           | Cycle reproducibility C1/C2, 6-cycle rule | 11,765               | 35                     | 0.3%                | 76.5%                 |
| healthy ovulatory              | 18 months           | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 24 months           | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 24 months           | Windowed Herzog C1/C2 union               | 49,381               | 7,939                  | 16.1%               | 1.2%                  |
| healthy ovulatory              | 24 months           | Windowed Herzog thresholds                | 49,381               | 7,939                  | 16.1%               | 1.2%                  |
| healthy ovulatory              | 24 months           | Windowed Herzog C1/C2 with minimum data   | 47,535               | 6,983                  | 14.7%               | 4.9%                  |
| healthy ovulatory              | 24 months           | Cycle reproducibility C1/C2, 6-cycle rule | 9,088                | 6                      | 0.1%                | 81.8%                 |
| healthy ovulatory              | 24 months           | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 3 cycles            | Exact Herzog 2004, any CE pattern         | 22,113               | 11,173                 | 50.5%               | 55.8%                 |
| healthy ovulatory              | 3 cycles            | Windowed Herzog C1/C2 union               | 45,069               | 18,699                 | 41.5%               | 9.9%                  |
| healthy ovulatory              | 3 cycles            | Windowed Herzog thresholds                | 45,069               | 18,699                 | 41.5%               | 9.9%                  |
| healthy ovulatory              | 3 cycles            | Windowed Herzog C1/C2 with minimum data   | 103                  | 31                     | 30.1%               | 99.8%                 |
| healthy ovulatory              | 3 cycles            | Cycle reproducibility C1/C2, 6-cycle rule | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 3 cycles            | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 3 months            | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 3 months            | Windowed Herzog C1/C2 union               | 45,321               | 18,868                 | 41.6%               | 9.4%                  |
| healthy ovulatory              | 3 months            | Windowed Herzog thresholds                | 45,321               | 18,868                 | 41.6%               | 9.4%                  |
| healthy ovulatory              | 3 months            | Windowed Herzog C1/C2 with minimum data   | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 3 months            | Cycle reproducibility C1/C2, 6-cycle rule | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 3 months            | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 36 months           | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 36 months           | Windowed Herzog C1/C2 union               | 49,605               | 5,631                  | 11.4%               | 0.8%                  |
| healthy ovulatory              | 36 months           | Windowed Herzog thresholds                | 49,605               | 5,631                  | 11.4%               | 0.8%                  |
| healthy ovulatory              | 36 months           | Windowed Herzog C1/C2 with minimum data   | 48,359               | 4,993                  | 10.3%               | 3.3%                  |
| healthy ovulatory              | 36 months           | Cycle reproducibility C1/C2, 6-cycle rule | 5,914                | 0                      | 0.0%                | 88.2%                 |
| healthy ovulatory              | 36 months           | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 36-month full diary | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 36-month full diary | Windowed Herzog C1/C2 union               | 49,605               | 5,631                  | 11.4%               | 0.8%                  |
| healthy ovulatory              | 36-month full diary | Windowed Herzog thresholds                | 49,605               | 5,631                  | 11.4%               | 0.8%                  |
| healthy ovulatory              | 36-month full diary | Windowed Herzog C1/C2 with minimum data   | 48,359               | 4,993                  | 10.3%               | 3.3%                  |
| healthy ovulatory              | 36-month full diary | Cycle reproducibility C1/C2, 6-cycle rule | 5,914                | 0                      | 0.0%                | 88.2%                 |
| healthy ovulatory              | 36-month full diary | Negative-binomial regression C1/C2        | 48,359               | 2,077                  | 4.3%                | 3.3%                  |
| healthy ovulatory              | 4 months            | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 4 months            | Windowed Herzog C1/C2 union               | 46,302               | 17,865                 | 38.6%               | 7.4%                  |
| healthy ovulatory              | 4 months            | Windowed Herzog thresholds                | 46,302               | 17,865                 | 38.6%               | 7.4%                  |
| healthy ovulatory              | 4 months            | Windowed Herzog C1/C2 with minimum data   | 37,627               | 13,437                 | 35.7%               | 24.7%                 |
| healthy ovulatory              | 4 months            | Cycle reproducibility C1/C2, 6-cycle rule | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 4 months            | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 6 cycles            | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 6 cycles            | Windowed Herzog C1/C2 union               | 47,427               | 16,467                 | 34.7%               | 5.1%                  |
| healthy ovulatory              | 6 cycles            | Windowed Herzog thresholds                | 47,427               | 16,467                 | 34.7%               | 5.1%                  |
| healthy ovulatory              | 6 cycles            | Windowed Herzog C1/C2 with minimum data   | 40,860               | 13,070                 | 32.0%               | 18.3%                 |
| healthy ovulatory              | 6 cycles            | Cycle reproducibility C1/C2, 6-cycle rule | 22,361               | 2,763                  | 12.4%               | 55.3%                 |
| healthy ovulatory              | 6 cycles            | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 6 months            | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 6 months            | Windowed Herzog C1/C2 union               | 47,533               | 16,138                 | 34.0%               | 4.9%                  |
| healthy ovulatory              | 6 months            | Windowed Herzog thresholds                | 47,533               | 16,138                 | 34.0%               | 4.9%                  |
| healthy ovulatory              | 6 months            | Windowed Herzog C1/C2 with minimum data   | 41,140               | 12,805                 | 31.1%               | 17.7%                 |
| healthy ovulatory              | 6 months            | Cycle reproducibility C1/C2, 6-cycle rule | 7,826                | 861                    | 11.0%               | 84.3%                 |
| healthy ovulatory              | 6 months            | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 9 months            | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 9 months            | Windowed Herzog C1/C2 union               | 48,331               | 14,193                 | 29.4%               | 3.3%                  |
| healthy ovulatory              | 9 months            | Windowed Herzog thresholds                | 48,331               | 14,193                 | 29.4%               | 3.3%                  |
| healthy ovulatory              | 9 months            | Windowed Herzog C1/C2 with minimum data   | 43,805               | 11,879                 | 27.1%               | 12.4%                 |
| healthy ovulatory              | 9 months            | Cycle reproducibility C1/C2, 6-cycle rule | 18,865               | 637                    | 3.4%                | 62.3%                 |
| healthy ovulatory              | 9 months            | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 1 month             | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 1 month             | Windowed Herzog C1/C2 union               | 37,972               | 18,310                 | 48.2%               | 24.1%                 |
| heterogeneous menstruating-age | 1 month             | Windowed Herzog thresholds                | 37,972               | 19,826                 | 52.2%               | 24.1%                 |
| heterogeneous menstruating-age | 1 month             | Windowed Herzog C1/C2 with minimum data   | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 1 month             | Cycle reproducibility C1/C2, 6-cycle rule | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 1 month             | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 12 cycles           | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 12 cycles           | Windowed Herzog C1/C2 union               | 48,720               | 12,205                 | 25.1%               | 2.6%                  |
| heterogeneous menstruating-age | 12 cycles           | Windowed Herzog thresholds                | 48,720               | 20,077                 | 41.2%               | 2.6%                  |
| heterogeneous menstruating-age | 12 cycles           | Windowed Herzog C1/C2 with minimum data   | 45,249               | 10,387                 | 23.0%               | 9.5%                  |
| heterogeneous menstruating-age | 12 cycles           | Cycle reproducibility C1/C2, 6-cycle rule | 18,313               | 137                    | 0.7%                | 63.4%                 |
| heterogeneous menstruating-age | 12 cycles           | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 12 months           | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 12 months           | Windowed Herzog C1/C2 union               | 48,719               | 12,352                 | 25.4%               | 2.6%                  |
| heterogeneous menstruating-age | 12 months           | Windowed Herzog thresholds                | 48,719               | 20,459                 | 42.0%               | 2.6%                  |
| heterogeneous menstruating-age | 12 months           | Windowed Herzog C1/C2 with minimum data   | 45,278               | 10,562                 | 23.3%               | 9.4%                  |
| heterogeneous menstruating-age | 12 months           | Cycle reproducibility C1/C2, 6-cycle rule | 19,110               | 120                    | 0.6%                | 61.8%                 |
| heterogeneous menstruating-age | 12 months           | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 18 months           | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 18 months           | Windowed Herzog C1/C2 union               | 49,139               | 9,863                  | 20.1%               | 1.7%                  |
| heterogeneous menstruating-age | 18 months           | Windowed Herzog thresholds                | 49,139               | 19,315                 | 39.3%               | 1.7%                  |
| heterogeneous menstruating-age | 18 months           | Windowed Herzog C1/C2 with minimum data   | 46,757               | 8,622                  | 18.4%               | 6.5%                  |
| heterogeneous menstruating-age | 18 months           | Cycle reproducibility C1/C2, 6-cycle rule | 14,954               | 14                     | 0.1%                | 70.1%                 |
| heterogeneous menstruating-age | 18 months           | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 24 months           | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 24 months           | Windowed Herzog C1/C2 union               | 49,373               | 7,990                  | 16.2%               | 1.3%                  |
| heterogeneous menstruating-age | 24 months           | Windowed Herzog thresholds                | 49,373               | 18,625                 | 37.7%               | 1.3%                  |
| heterogeneous menstruating-age | 24 months           | Windowed Herzog C1/C2 with minimum data   | 47,514               | 7,009                  | 14.8%               | 5.0%                  |
| heterogeneous menstruating-age | 24 months           | Cycle reproducibility C1/C2, 6-cycle rule | 12,313               | 1                      | 0.0%                | 75.4%                 |
| heterogeneous menstruating-age | 24 months           | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 3 cycles            | Exact Herzog 2004, any CE pattern         | 18,153               | 9,482                  | 52.2%               | 63.7%                 |
| heterogeneous menstruating-age | 3 cycles            | Windowed Herzog C1/C2 union               | 45,206               | 18,676                 | 41.3%               | 9.6%                  |
| heterogeneous menstruating-age | 3 cycles            | Windowed Herzog thresholds                | 45,206               | 22,844                 | 50.5%               | 9.6%                  |
| heterogeneous menstruating-age | 3 cycles            | Windowed Herzog C1/C2 with minimum data   | 1,587                | 631                    | 39.8%               | 96.8%                 |
| heterogeneous menstruating-age | 3 cycles            | Cycle reproducibility C1/C2, 6-cycle rule | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 3 cycles            | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 3 months            | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 3 months            | Windowed Herzog C1/C2 union               | 45,194               | 19,097                 | 42.3%               | 9.6%                  |
| heterogeneous menstruating-age | 3 months            | Windowed Herzog thresholds                | 45,194               | 23,037                 | 51.0%               | 9.6%                  |
| heterogeneous menstruating-age | 3 months            | Windowed Herzog C1/C2 with minimum data   | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 3 months            | Cycle reproducibility C1/C2, 6-cycle rule | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 3 months            | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 36 months           | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 36 months           | Windowed Herzog C1/C2 union               | 49,587               | 5,742                  | 11.6%               | 0.8%                  |
| heterogeneous menstruating-age | 36 months           | Windowed Herzog thresholds                | 49,587               | 17,774                 | 35.8%               | 0.8%                  |
| heterogeneous menstruating-age | 36 months           | Windowed Herzog C1/C2 with minimum data   | 48,333               | 5,070                  | 10.5%               | 3.3%                  |
| heterogeneous menstruating-age | 36 months           | Cycle reproducibility C1/C2, 6-cycle rule | 8,838                | 0                      | 0.0%                | 82.3%                 |
| heterogeneous menstruating-age | 36 months           | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 36-month full diary | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 36-month full diary | Windowed Herzog C1/C2 union               | 49,587               | 5,742                  | 11.6%               | 0.8%                  |
| heterogeneous menstruating-age | 36-month full diary | Windowed Herzog thresholds                | 49,587               | 17,774                 | 35.8%               | 0.8%                  |
| heterogeneous menstruating-age | 36-month full diary | Windowed Herzog C1/C2 with minimum data   | 48,333               | 5,070                  | 10.5%               | 3.3%                  |
| heterogeneous menstruating-age | 36-month full diary | Cycle reproducibility C1/C2, 6-cycle rule | 8,838                | 0                      | 0.0%                | 82.3%                 |
| heterogeneous menstruating-age | 36-month full diary | Negative-binomial regression C1/C2        | 48,333               | 2,063                  | 4.3%                | 3.3%                  |
| heterogeneous menstruating-age | 4 months            | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 4 months            | Windowed Herzog C1/C2 union               | 46,301               | 17,999                 | 38.9%               | 7.4%                  |
| heterogeneous menstruating-age | 4 months            | Windowed Herzog thresholds                | 46,301               | 22,911                 | 49.5%               | 7.4%                  |
| heterogeneous menstruating-age | 4 months            | Windowed Herzog C1/C2 with minimum data   | 37,366               | 13,449                 | 36.0%               | 25.3%                 |
| heterogeneous menstruating-age | 4 months            | Cycle reproducibility C1/C2, 6-cycle rule | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 4 months            | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 6 cycles            | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 6 cycles            | Windowed Herzog C1/C2 union               | 47,477               | 16,312                 | 34.4%               | 5.0%                  |
| heterogeneous menstruating-age | 6 cycles            | Windowed Herzog thresholds                | 47,477               | 22,168                 | 46.7%               | 5.0%                  |
| heterogeneous menstruating-age | 6 cycles            | Windowed Herzog C1/C2 with minimum data   | 41,115               | 12,977                 | 31.6%               | 17.8%                 |
| heterogeneous menstruating-age | 6 cycles            | Cycle reproducibility C1/C2, 6-cycle rule | 25,178               | 1,630                  | 6.5%                | 49.6%                 |
| heterogeneous menstruating-age | 6 cycles            | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 6 months            | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 6 months            | Windowed Herzog C1/C2 union               | 47,453               | 16,497                 | 34.8%               | 5.1%                  |
| heterogeneous menstruating-age | 6 months            | Windowed Herzog thresholds                | 47,453               | 22,360                 | 47.1%               | 5.1%                  |
| heterogeneous menstruating-age | 6 months            | Windowed Herzog C1/C2 with minimum data   | 41,163               | 13,266                 | 32.2%               | 17.7%                 |
| heterogeneous menstruating-age | 6 months            | Cycle reproducibility C1/C2, 6-cycle rule | 6,772                | 510                    | 7.5%                | 86.5%                 |
| heterogeneous menstruating-age | 6 months            | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 9 months            | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 9 months            | Windowed Herzog C1/C2 union               | 48,306               | 14,169                 | 29.3%               | 3.4%                  |
| heterogeneous menstruating-age | 9 months            | Windowed Herzog thresholds                | 48,306               | 21,339                 | 44.2%               | 3.4%                  |
| heterogeneous menstruating-age | 9 months            | Windowed Herzog C1/C2 with minimum data   | 43,903               | 11,850                 | 27.0%               | 12.2%                 |
| heterogeneous menstruating-age | 9 months            | Cycle reproducibility C1/C2, 6-cycle rule | 21,508               | 353                    | 1.6%                | 57.0%                 |
| heterogeneous menstruating-age | 9 months            | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |

**Table 3 caption.** False-positive and indeterminate rates for every prespecified observation window and core definition. Exact Herzog 2004 is expected to be classifiable only for 3-complete-cycle windows.

## Table 4. Strict Herzog versus luteal-anchored ovulatory sensitivity

**Why this table is included.** This table addresses whether false-positive rates depend on the strict Herzog periovulatory window expanding with cycle length. Strict Herzog remains primary for historical comparability; the luteal-anchored mode fixes the ovulatory window at four pre-luteal days.

**Code to call.**

In [6]:
phase_mode_sensitivity = summary_tables[
    (summary_tables.table_type == "window_false_positive")
    & (summary_tables.subset == "all")
    & (
        (summary_tables.window_type == "full")
        | ((summary_tables.window_type == "calendar") & (summary_tables.window_value.astype(str) == "3"))
    )
    & (summary_tables.definition.isin(["A_windowed_any", "A_windowed_C1_or_C2", "A_windowed_C3_only"]))
].copy()
phase_mode_sensitivity


,table_type,subset,cohort,window_type,window_value,definition,phase_mode,assumption_based_historical,n_windows,n_classifiable,...,indeterminate_rate,positive_rate_all_attempted,unstable_denominator,interpretation_note,n_participants,pattern_category,indeterminate_reason,n_indeterminate,p_prevalence_ge_39_1,p_prevalence_ge_44_2
53,window_false_positive,all,healthy_ovulatory,calendar,3,A_windowed_any,luteal_anchored_ovulatory,False,50000,45321.0,...,0.09358,0.36318,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
64,window_false_positive,all,healthy_ovulatory,full,full_diary,A_windowed_any,luteal_anchored_ovulatory,False,50000,49605.0,...,0.00790,0.11978,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
66,window_false_positive,all,healthy_ovulatory,calendar,3,A_windowed_any,strict_herzog,False,50000,45321.0,...,0.09358,0.37736,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
77,window_false_positive,all,healthy_ovulatory,full,full_diary,A_windowed_any,strict_herzog,False,50000,49605.0,...,0.00790,0.11262,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
79,window_false_positive,all,population,calendar,3,A_windowed_any,luteal_anchored_ovulatory,False,50000,45189.0,...,0.09622,0.42316,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
90,window_false_positive,all,population,full,full_diary,A_windowed_any,luteal_anchored_ovulatory,False,50000,49587.0,...,0.00826,0.29852,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
92,window_false_positive,all,population,calendar,3,A_windowed_any,strict_herzog,False,50000,45194.0,...,0.09612,0.46074,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
103,window_false_positive,all,population,full,full_diary,A_windowed_any,strict_herzog,False,50000,49587.0,...,0.00826,0.35548,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
157,window_false_positive,all,healthy_ovulatory,calendar,3,A_windowed_C1_or_C2,luteal_anchored_ovulatory,False,50000,45321.0,...,0.09358,0.36318,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
168,window_false_positive,all,healthy_ovulatory,full,full_diary,A_windowed_C1_or_C2,luteal_anchored_ovulatory,False,50000,49605.0,...,0.00790,0.11978,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


| Cohort                         | Phase labeling            | Observation window  | CE definition               | Classifiable windows | False-positive windows | False-positive rate (95% CI) | Indeterminate windows |
| ------------------------------ | ------------------------- | ------------------- | --------------------------- | -------------------- | ---------------------- | ---------------------------- | --------------------- |
| healthy ovulatory              | Luteal-anchored ovulatory | 3 months            | Windowed Herzog C1/C2 union | 45,321               | 18,159                 | 40.1% (39.6, 40.5)           | 9.4%                  |
| healthy ovulatory              | Strict Herzog             | 3 months            | Windowed Herzog C1/C2 union | 45,321               | 18,868                 | 41.6% (41.2, 42.1)           | 9.4%                  |
| healthy ovulatory              | Luteal-anchored ovulatory | 3 months            | Windowed Herzog C3 only     | 0                    | 0                      | NA                           | 100.0%                |
| healthy ovulatory              | Strict Herzog             | 3 months            | Windowed Herzog C3 only     | 0                    | 0                      | NA                           | 100.0%                |
| healthy ovulatory              | Luteal-anchored ovulatory | 3 months            | Windowed Herzog thresholds  | 45,321               | 18,159                 | 40.1% (39.6, 40.5)           | 9.4%                  |
| healthy ovulatory              | Strict Herzog             | 3 months            | Windowed Herzog thresholds  | 45,321               | 18,868                 | 41.6% (41.2, 42.1)           | 9.4%                  |
| healthy ovulatory              | Luteal-anchored ovulatory | 36-month full diary | Windowed Herzog C1/C2 union | 49,605               | 5,989                  | 12.1% (11.8, 12.4)           | 0.8%                  |
| healthy ovulatory              | Strict Herzog             | 36-month full diary | Windowed Herzog C1/C2 union | 49,605               | 5,631                  | 11.4% (11.1, 11.6)           | 0.8%                  |
| healthy ovulatory              | Luteal-anchored ovulatory | 36-month full diary | Windowed Herzog C3 only     | 0                    | 0                      | NA                           | 100.0%                |
| healthy ovulatory              | Strict Herzog             | 36-month full diary | Windowed Herzog C3 only     | 0                    | 0                      | NA                           | 100.0%                |
| healthy ovulatory              | Luteal-anchored ovulatory | 36-month full diary | Windowed Herzog thresholds  | 49,605               | 5,989                  | 12.1% (11.8, 12.4)           | 0.8%                  |
| healthy ovulatory              | Strict Herzog             | 36-month full diary | Windowed Herzog thresholds  | 49,605               | 5,631                  | 11.4% (11.1, 11.6)           | 0.8%                  |
| heterogeneous menstruating-age | Luteal-anchored ovulatory | 3 months            | Windowed Herzog C1/C2 union | 45,189               | 18,219                 | 40.3% (39.9, 40.8)           | 9.6%                  |
| heterogeneous menstruating-age | Strict Herzog             | 3 months            | Windowed Herzog C1/C2 union | 45,194               | 19,097                 | 42.3% (41.8, 42.7)           | 9.6%                  |
| heterogeneous menstruating-age | Luteal-anchored ovulatory | 3 months            | Windowed Herzog C3 only     | 16,543               | 6,680                  | 40.4% (39.6, 41.1)           | 66.9%                 |
| heterogeneous menstruating-age | Strict Herzog             | 3 months            | Windowed Herzog C3 only     | 16,141               | 8,648                  | 53.6% (52.8, 54.3)           | 67.7%                 |
| heterogeneous menstruating-age | Luteal-anchored ovulatory | 3 months            | Windowed Herzog thresholds  | 45,189               | 21,158                 | 46.8% (46.4, 47.3)           | 9.6%                  |
| heterogeneous menstruating-age | Strict Herzog             | 3 months            | Windowed Herzog thresholds  | 45,194               | 23,037                 | 51.0% (50.5, 51.4)           | 9.6%                  |
| heterogeneous menstruating-age | Luteal-anchored ovulatory | 36-month full diary | Windowed Herzog C1/C2 union | 49,587               | 6,133                  | 12.4% (12.1, 12.7)           | 0.8%                  |
| heterogeneous menstruating-age | Strict Herzog             | 36-month full diary | Windowed Herzog C1/C2 union | 49,587               | 5,742                  | 11.6% (11.3, 11.9)           | 0.8%                  |
| heterogeneous menstruating-age | Luteal-anchored ovulatory | 36-month full diary | Windowed Herzog C3 only     | 38,603               | 10,738                 | 27.8% (27.4, 28.3)           | 22.8%                 |
| heterogeneous menstruating-age | Strict Herzog             | 36-month full diary | Windowed Herzog C3 only     | 38,583               | 14,293                 | 37.0% (36.6, 37.5)           | 22.8%                 |
| heterogeneous menstruating-age | Luteal-anchored ovulatory | 36-month full diary | Windowed Herzog thresholds  | 49,587               | 14,926                 | 30.1% (29.7, 30.5)           | 0.8%                  |
| heterogeneous menstruating-age | Strict Herzog             | 36-month full diary | Windowed Herzog thresholds  | 49,587               | 17,774                 | 35.8% (35.4, 36.3)           | 0.8%                  |

**Table 4 caption.** Full-diary and 3-month windowed Herzog results under strict Herzog and luteal-anchored ovulatory phase labeling.

## Table 5. Null study-level prevalence benchmarks

**Why this table is included.** This table maps person-level false positives into apparent prevalence in illustrative studies of 30, 50, and 100 participants. It reports prevalence among all participants, prevalence among classifiable participants only, and the probability of exceeding the 39.1% and 44.2% benchmark values.

**Code to call.**

In [7]:
study_benchmarks = summary_tables[
    (summary_tables.table_type == "study_level_3month")
    & (summary_tables.phase_mode == "strict_herzog")
    & (summary_tables.definition.isin([
        "A_windowed_any", "B_minimum_data_C1_or_C2",
        "C_reproducibility_C1_or_C2", "D_nb_regression_C1_or_C2"
    ]))
].copy()
study_benchmarks


,table_type,subset,cohort,window_type,window_value,definition,phase_mode,assumption_based_historical,n_windows,n_classifiable,...,indeterminate_rate,positive_rate_all_attempted,unstable_denominator,interpretation_note,n_participants,pattern_category,indeterminate_reason,n_indeterminate,p_prevalence_ge_39_1,p_prevalence_ge_44_2
19457,study_level_3month,apparent_prevalence_all,healthy_ovulatory,study_mc_calendar,3,A_windowed_any,strict_herzog,False,10000,10000.0,...,NaN,0.376547,False,NaN,30.0,NaN,NaN,NaN,0.4689,0.2031
19458,study_level_3month,apparent_prevalence_classifiable,healthy_ovulatory,study_mc_calendar,3,A_windowed_any,strict_herzog,False,10000,10000.0,...,NaN,0.416039,False,NaN,30.0,NaN,NaN,NaN,0.6131,0.4041
19459,study_level_3month,apparent_prevalence_all,healthy_ovulatory,study_mc_calendar,3,A_windowed_any,strict_herzog,False,10000,10000.0,...,NaN,0.377548,False,NaN,50.0,NaN,NaN,NaN,0.4232,0.1527
19460,study_level_3month,apparent_prevalence_classifiable,healthy_ovulatory,study_mc_calendar,3,A_windowed_any,strict_herzog,False,10000,10000.0,...,NaN,0.416880,False,NaN,50.0,NaN,NaN,NaN,0.6385,0.3611
19461,study_level_3month,apparent_prevalence_all,healthy_ovulatory,study_mc_calendar,3,A_windowed_any,strict_herzog,False,10000,10000.0,...,NaN,0.375478,False,NaN,100.0,NaN,NaN,NaN,0.3398,0.0784
19462,study_level_3month,apparent_prevalence_classifiable,healthy_ovulatory,study_mc_calendar,3,A_windowed_any,strict_herzog,False,10000,10000.0,...,NaN,0.415031,False,NaN,100.0,NaN,NaN,NaN,0.6754,0.2996
19469,study_level_3month,apparent_prevalence_all,healthy_ovulatory,study_mc_calendar,3,B_minimum_data_C1_or_C2,strict_herzog,False,10000,10000.0,...,NaN,0.000000,False,NaN,30.0,NaN,NaN,NaN,0.0000,0.0000
19470,study_level_3month,apparent_prevalence_classifiable,healthy_ovulatory,study_mc_calendar,3,B_minimum_data_C1_or_C2,strict_herzog,False,0,0.0,...,NaN,NaN,False,NaN,30.0,NaN,NaN,NaN,NaN,NaN
19471,study_level_3month,apparent_prevalence_all,healthy_ovulatory,study_mc_calendar,3,B_minimum_data_C1_or_C2,strict_herzog,False,10000,10000.0,...,NaN,0.000000,False,NaN,50.0,NaN,NaN,NaN,0.0000,0.0000
19472,study_level_3month,apparent_prevalence_classifiable,healthy_ovulatory,study_mc_calendar,3,B_minimum_data_C1_or_C2,strict_herzog,False,0,0.0,...,NaN,NaN,False,NaN,50.0,NaN,NaN,NaN,NaN,NaN


| Cohort                         | CE definition                             | Participants per study | Analysis denominator           | Monte Carlo studies | Mean apparent CE prevalence | 2.5th percentile | 97.5th percentile | Probability prevalence at least 39.1% | Probability prevalence at least 44.2% |
| ------------------------------ | ----------------------------------------- | ---------------------- | ------------------------------ | ------------------- | --------------------------- | ---------------- | ----------------- | ------------------------------------- | ------------------------------------- |
| healthy ovulatory              | Windowed Herzog thresholds                | 30.0                   | All participants               | 10,000              | 37.7%                       | 20.0%            | 56.7%             | 46.9%                                 | 20.3%                                 |
| healthy ovulatory              | Windowed Herzog thresholds                | 30.0                   | Classifiable participants only | 10,000              | 41.6%                       | 23.1%            | 60.0%             | 61.3%                                 | 40.4%                                 |
| healthy ovulatory              | Windowed Herzog thresholds                | 50.0                   | All participants               | 10,000              | 37.8%                       | 24.0%            | 52.0%             | 42.3%                                 | 15.3%                                 |
| healthy ovulatory              | Windowed Herzog thresholds                | 50.0                   | Classifiable participants only | 10,000              | 41.7%                       | 27.5%            | 56.5%             | 63.8%                                 | 36.1%                                 |
| healthy ovulatory              | Windowed Herzog thresholds                | 100.0                  | All participants               | 10,000              | 37.5%                       | 28.0%            | 47.0%             | 34.0%                                 | 7.8%                                  |
| healthy ovulatory              | Windowed Herzog thresholds                | 100.0                  | Classifiable participants only | 10,000              | 41.5%                       | 31.5%            | 51.7%             | 67.5%                                 | 30.0%                                 |
| healthy ovulatory              | Windowed Herzog C1/C2 with minimum data   | 30.0                   | All participants               | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| healthy ovulatory              | Windowed Herzog C1/C2 with minimum data   | 30.0                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| healthy ovulatory              | Windowed Herzog C1/C2 with minimum data   | 50.0                   | All participants               | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| healthy ovulatory              | Windowed Herzog C1/C2 with minimum data   | 50.0                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| healthy ovulatory              | Windowed Herzog C1/C2 with minimum data   | 100.0                  | All participants               | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| healthy ovulatory              | Windowed Herzog C1/C2 with minimum data   | 100.0                  | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| healthy ovulatory              | Cycle reproducibility C1/C2, 6-cycle rule | 30.0                   | All participants               | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| healthy ovulatory              | Cycle reproducibility C1/C2, 6-cycle rule | 30.0                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| healthy ovulatory              | Cycle reproducibility C1/C2, 6-cycle rule | 50.0                   | All participants               | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| healthy ovulatory              | Cycle reproducibility C1/C2, 6-cycle rule | 50.0                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| healthy ovulatory              | Cycle reproducibility C1/C2, 6-cycle rule | 100.0                  | All participants               | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| healthy ovulatory              | Cycle reproducibility C1/C2, 6-cycle rule | 100.0                  | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| healthy ovulatory              | Negative-binomial regression C1/C2        | 30.0                   | All participants               | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| healthy ovulatory              | Negative-binomial regression C1/C2        | 30.0                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| healthy ovulatory              | Negative-binomial regression C1/C2        | 50.0                   | All participants               | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| healthy ovulatory              | Negative-binomial regression C1/C2        | 50.0                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| healthy ovulatory              | Negative-binomial regression C1/C2        | 100.0                  | All participants               | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| healthy ovulatory              | Negative-binomial regression C1/C2        | 100.0                  | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| heterogeneous menstruating-age | Windowed Herzog thresholds                | 30.0                   | All participants               | 10,000              | 46.0%                       | 30.0%            | 63.3%             | 80.1%                                 | 53.8%                                 |
| heterogeneous menstruating-age | Windowed Herzog thresholds                | 30.0                   | Classifiable participants only | 10,000              | 50.8%                       | 32.1%            | 70.0%             | 89.5%                                 | 75.6%                                 |
| heterogeneous menstruating-age | Windowed Herzog thresholds                | 50.0                   | All participants               | 10,000              | 46.0%                       | 32.0%            | 60.0%             | 84.3%                                 | 56.0%                                 |
| heterogeneous menstruating-age | Windowed Herzog thresholds                | 50.0                   | Classifiable participants only | 10,000              | 50.8%                       | 36.4%            | 65.2%             | 94.1%                                 | 81.6%                                 |
| heterogeneous menstruating-age | Windowed Herzog thresholds                | 100.0                  | All participants               | 10,000              | 46.1%                       | 36.0%            | 56.0%             | 90.6%                                 | 62.1%                                 |
| heterogeneous menstruating-age | Windowed Herzog thresholds                | 100.0                  | Classifiable participants only | 10,000              | 50.9%                       | 40.7%            | 61.3%             | 98.8%                                 | 89.7%                                 |
| heterogeneous menstruating-age | Windowed Herzog C1/C2 with minimum data   | 30.0                   | All participants               | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| heterogeneous menstruating-age | Windowed Herzog C1/C2 with minimum data   | 30.0                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| heterogeneous menstruating-age | Windowed Herzog C1/C2 with minimum data   | 50.0                   | All participants               | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| heterogeneous menstruating-age | Windowed Herzog C1/C2 with minimum data   | 50.0                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| heterogeneous menstruating-age | Windowed Herzog C1/C2 with minimum data   | 100.0                  | All participants               | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| heterogeneous menstruating-age | Windowed Herzog C1/C2 with minimum data   | 100.0                  | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| heterogeneous menstruating-age | Cycle reproducibility C1/C2, 6-cycle rule | 30.0                   | All participants               | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| heterogeneous menstruating-age | Cycle reproducibility C1/C2, 6-cycle rule | 30.0                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| heterogeneous menstruating-age | Cycle reproducibility C1/C2, 6-cycle rule | 50.0                   | All participants               | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| heterogeneous menstruating-age | Cycle reproducibility C1/C2, 6-cycle rule | 50.0                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| heterogeneous menstruating-age | Cycle reproducibility C1/C2, 6-cycle rule | 100.0                  | All participants               | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| heterogeneous menstruating-age | Cycle reproducibility C1/C2, 6-cycle rule | 100.0                  | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| heterogeneous menstruating-age | Negative-binomial regression C1/C2        | 30.0                   | All participants               | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| heterogeneous menstruating-age | Negative-binomial regression C1/C2        | 30.0                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| heterogeneous menstruating-age | Negative-binomial regression C1/C2        | 50.0                   | All participants               | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| heterogeneous menstruating-age | Negative-binomial regression C1/C2        | 50.0                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| heterogeneous menstruating-age | Negative-binomial regression C1/C2        | 100.0                  | All participants               | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| heterogeneous menstruating-age | Negative-binomial regression C1/C2        | 100.0                  | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |

**Table 5 caption.** Study-level Monte Carlo summary from null studies using 3-month windows. The interval columns are the 2.5th and 97.5th percentiles of study-level apparent prevalence.

## Table 6. Trial-like conditioned subsets

**Why this table is included.** These subsets answer whether common enrollment restrictions reduce false positives or mainly change the classifiable denominator. The common-classifiable subset supports head-to-head comparisons because every listed core definition is defined on the same windows.

**Code to call.**

In [8]:
trial_like_subsets = summary_tables[
    (summary_tables.table_type == "window_false_positive")
    & (summary_tables.phase_mode == "strict_herzog")
    & (summary_tables.window_type == "full")
    & (summary_tables.subset.isin([
        "ge_1_seizure_day_per_month",
        "ge_2_seizures_per_month",
        "strict_23_35_day_cycles_only",
        "common_classifiable_subset",
    ]))
    & (summary_tables.definition.isin([
        "A_windowed_any", "A_windowed_C1_or_C2",
        "B_minimum_data_C1_or_C2",
        "C_reproducibility_C1_or_C2", "D_nb_regression_C1_or_C2"
    ]))
].copy()
trial_like_subsets


,table_type,subset,cohort,window_type,window_value,definition,phase_mode,assumption_based_historical,n_windows,n_classifiable,...,indeterminate_rate,positive_rate_all_attempted,unstable_denominator,interpretation_note,n_participants,pattern_category,indeterminate_reason,n_indeterminate,p_prevalence_ge_39_1,p_prevalence_ge_44_2
1273,window_false_positive,ge_1_seizure_day_per_month,healthy_ovulatory,full,full_diary,A_windowed_any,strict_herzog,False,37309,37309.0,...,0.000000,0.053124,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1299,window_false_positive,ge_1_seizure_day_per_month,population,full,full_diary,A_windowed_any,strict_herzog,False,37196,37196.0,...,0.000000,0.310705,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1377,window_false_positive,ge_1_seizure_day_per_month,healthy_ovulatory,full,full_diary,A_windowed_C1_or_C2,strict_herzog,False,37309,37309.0,...,0.000000,0.053124,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1403,window_false_positive,ge_1_seizure_day_per_month,population,full,full_diary,A_windowed_C1_or_C2,strict_herzog,False,37196,37196.0,...,0.000000,0.054710,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1689,window_false_positive,ge_1_seizure_day_per_month,healthy_ovulatory,full,full_diary,B_minimum_data_C1_or_C2,strict_herzog,False,37309,37309.0,...,0.000000,0.053124,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1715,window_false_positive,ge_1_seizure_day_per_month,population,full,full_diary,B_minimum_data_C1_or_C2,strict_herzog,False,37196,37196.0,...,0.000000,0.054710,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1793,window_false_positive,ge_1_seizure_day_per_month,healthy_ovulatory,full,full_diary,C_reproducibility_C1_or_C2,strict_herzog,False,37309,5914.0,...,0.841486,0.000000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1819,window_false_positive,ge_1_seizure_day_per_month,population,full,full_diary,C_reproducibility_C1_or_C2,strict_herzog,False,37196,8831.0,...,0.762582,0.000000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2001,window_false_positive,ge_1_seizure_day_per_month,healthy_ovulatory,full,full_diary,D_nb_regression_C1_or_C2,strict_herzog,False,37309,37309.0,...,0.000000,0.045592,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2027,window_false_positive,ge_1_seizure_day_per_month,population,full,full_diary,D_nb_regression_C1_or_C2,strict_herzog,False,37196,37196.0,...,0.000000,0.045489,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


| Cohort                         | Analysis denominator             | CE definition                             | Classifiable windows | False-positive windows | False-positive rate (95% CI) | Indeterminate windows |
| ------------------------------ | -------------------------------- | ----------------------------------------- | -------------------- | ---------------------- | ---------------------------- | --------------------- |
| healthy ovulatory              | Common classifiable subset       | Windowed Herzog C1/C2 union               | 5,914                | 111                    | 1.9% (1.6, 2.3)              | 0.0%                  |
| healthy ovulatory              | Common classifiable subset       | Windowed Herzog thresholds                | 5,914                | 111                    | 1.9% (1.6, 2.3)              | 0.0%                  |
| healthy ovulatory              | Common classifiable subset       | Windowed Herzog C1/C2 with minimum data   | 5,914                | 111                    | 1.9% (1.6, 2.3)              | 0.0%                  |
| healthy ovulatory              | Common classifiable subset       | Cycle reproducibility C1/C2, 6-cycle rule | 5,914                | 0                      | 0.0% (0.0, 0.1)              | 0.0%                  |
| healthy ovulatory              | Common classifiable subset       | Negative-binomial regression C1/C2        | 5,914                | 150                    | 2.5% (2.2, 3.0)              | 0.0%                  |
| healthy ovulatory              | At least 1 seizure day per month | Windowed Herzog C1/C2 union               | 37,309               | 1,982                  | 5.3% (5.1, 5.5)              | 0.0%                  |
| healthy ovulatory              | At least 1 seizure day per month | Windowed Herzog thresholds                | 37,309               | 1,982                  | 5.3% (5.1, 5.5)              | 0.0%                  |
| healthy ovulatory              | At least 1 seizure day per month | Windowed Herzog C1/C2 with minimum data   | 37,309               | 1,982                  | 5.3% (5.1, 5.5)              | 0.0%                  |
| healthy ovulatory              | At least 1 seizure day per month | Cycle reproducibility C1/C2, 6-cycle rule | 5,914                | 0                      | 0.0% (0.0, 0.1)              | 84.1%                 |
| healthy ovulatory              | At least 1 seizure day per month | Negative-binomial regression C1/C2        | 37,309               | 1,701                  | 4.6% (4.4, 4.8)              | 0.0%                  |
| healthy ovulatory              | At least 2 seizures per month    | Windowed Herzog C1/C2 union               | 30,106               | 1,203                  | 4.0% (3.8, 4.2)              | 0.0%                  |
| healthy ovulatory              | At least 2 seizures per month    | Windowed Herzog thresholds                | 30,106               | 1,203                  | 4.0% (3.8, 4.2)              | 0.0%                  |
| healthy ovulatory              | At least 2 seizures per month    | Windowed Herzog C1/C2 with minimum data   | 30,106               | 1,203                  | 4.0% (3.8, 4.2)              | 0.0%                  |
| healthy ovulatory              | At least 2 seizures per month    | Cycle reproducibility C1/C2, 6-cycle rule | 5,914                | 0                      | 0.0% (0.0, 0.1)              | 80.4%                 |
| healthy ovulatory              | At least 2 seizures per month    | Negative-binomial regression C1/C2        | 30,106               | 1,323                  | 4.4% (4.2, 4.6)              | 0.0%                  |
| healthy ovulatory              | Strict 23-35 day cycles only     | Windowed Herzog C1/C2 union               | 14,294               | 1,656                  | 11.6% (11.1, 12.1)           | 0.8%                  |
| healthy ovulatory              | Strict 23-35 day cycles only     | Windowed Herzog thresholds                | 14,294               | 1,656                  | 11.6% (11.1, 12.1)           | 0.8%                  |
| healthy ovulatory              | Strict 23-35 day cycles only     | Windowed Herzog C1/C2 with minimum data   | 13,937               | 1,464                  | 10.5% (10.0, 11.0)           | 3.3%                  |
| healthy ovulatory              | Strict 23-35 day cycles only     | Cycle reproducibility C1/C2, 6-cycle rule | 1,928                | 0                      | 0.0% (0.0, 0.2)              | 86.6%                 |
| healthy ovulatory              | Strict 23-35 day cycles only     | Negative-binomial regression C1/C2        | 13,937               | 594                    | 4.3% (3.9, 4.6)              | 3.3%                  |
| heterogeneous menstruating-age | Common classifiable subset       | Windowed Herzog C1/C2 union               | 12,710               | 594                    | 4.7% (4.3, 5.1)              | 0.0%                  |
| heterogeneous menstruating-age | Common classifiable subset       | Windowed Herzog thresholds                | 12,710               | 3,397                  | 26.7% (26.0, 27.5)           | 0.0%                  |
| heterogeneous menstruating-age | Common classifiable subset       | Windowed Herzog C1/C2 with minimum data   | 12,710               | 594                    | 4.7% (4.3, 5.1)              | 0.0%                  |
| heterogeneous menstruating-age | Common classifiable subset       | Cycle reproducibility C1/C2, 6-cycle rule | 8,838                | 0                      | 0.0% (0.0, 0.0)              | 30.5%                 |
| heterogeneous menstruating-age | Common classifiable subset       | Negative-binomial regression C1/C2        | 12,710               | 545                    | 4.3% (3.9, 4.7)              | 0.0%                  |
| heterogeneous menstruating-age | At least 1 seizure day per month | Windowed Herzog C1/C2 union               | 37,196               | 2,035                  | 5.5% (5.2, 5.7)              | 0.0%                  |
| heterogeneous menstruating-age | At least 1 seizure day per month | Windowed Herzog thresholds                | 37,196               | 11,557                 | 31.1% (30.6, 31.5)           | 0.0%                  |
| heterogeneous menstruating-age | At least 1 seizure day per month | Windowed Herzog C1/C2 with minimum data   | 37,196               | 2,035                  | 5.5% (5.2, 5.7)              | 0.0%                  |
| heterogeneous menstruating-age | At least 1 seizure day per month | Cycle reproducibility C1/C2, 6-cycle rule | 8,831                | 0                      | 0.0% (0.0, 0.0)              | 76.3%                 |
| heterogeneous menstruating-age | At least 1 seizure day per month | Negative-binomial regression C1/C2        | 37,196               | 1,692                  | 4.5% (4.3, 4.8)              | 0.0%                  |
| heterogeneous menstruating-age | At least 2 seizures per month    | Windowed Herzog C1/C2 union               | 30,016               | 1,273                  | 4.2% (4.0, 4.5)              | 0.0%                  |
| heterogeneous menstruating-age | At least 2 seizures per month    | Windowed Herzog thresholds                | 30,016               | 8,837                  | 29.4% (28.9, 30.0)           | 0.0%                  |
| heterogeneous menstruating-age | At least 2 seizures per month    | Windowed Herzog C1/C2 with minimum data   | 30,016               | 1,273                  | 4.2% (4.0, 4.5)              | 0.0%                  |
| heterogeneous menstruating-age | At least 2 seizures per month    | Cycle reproducibility C1/C2, 6-cycle rule | 8,794                | 0                      | 0.0% (0.0, 0.0)              | 70.7%                 |
| heterogeneous menstruating-age | At least 2 seizures per month    | Negative-binomial regression C1/C2        | 30,016               | 1,357                  | 4.5% (4.3, 4.8)              | 0.0%                  |
| heterogeneous menstruating-age | Strict 23-35 day cycles only     | Windowed Herzog C1/C2 union               | 10,465               | 1,169                  | 11.2% (10.6, 11.8)           | 0.9%                  |
| heterogeneous menstruating-age | Strict 23-35 day cycles only     | Windowed Herzog thresholds                | 10,465               | 4,082                  | 39.0% (38.1, 39.9)           | 0.9%                  |
| heterogeneous menstruating-age | Strict 23-35 day cycles only     | Windowed Herzog C1/C2 with minimum data   | 10,229               | 1,049                  | 10.3% (9.7, 10.9)            | 3.1%                  |
| heterogeneous menstruating-age | Strict 23-35 day cycles only     | Cycle reproducibility C1/C2, 6-cycle rule | 1,599                | 0                      | 0.0% (0.0, 0.2)              | 84.9%                 |
| heterogeneous menstruating-age | Strict 23-35 day cycles only     | Negative-binomial regression C1/C2        | 10,229               | 461                    | 4.5% (4.1, 4.9)              | 3.1%                  |

**Table 6 caption.** Full-diary false-positive rates after applying trial-like eligibility restrictions or a common classifiable denominator. This separates changes in apparent risk from changes in analyzability.

## Table 7. C1/C2/C3 decomposition and C3 exclusion

**Why this table is included.** This table directly addresses whether the heterogeneous-cohort signal is driven by C3 logic. It reports mutually exclusive pattern categories and C1/C2 union comparisons for full-diary windows.

**Code to call.**

In [9]:
pattern_decomposition = summary_tables[
    (summary_tables.table_type == "pattern_decomposition")
    & (summary_tables.phase_mode == "strict_herzog")
    & (summary_tables.window_type == "full")
    & (summary_tables.definition.isin(["A_windowed", "B_minimum_data", "D_nb_regression"]))
].copy()
pattern_decomposition


,table_type,subset,cohort,window_type,window_value,definition,phase_mode,assumption_based_historical,n_windows,n_classifiable,...,indeterminate_rate,positive_rate_all_attempted,unstable_denominator,interpretation_note,n_participants,pattern_category,indeterminate_reason,n_indeterminate,p_prevalence_ge_39_1,p_prevalence_ge_44_2
18044,pattern_decomposition,mutually_exclusive_patterns,healthy_ovulatory,full,full_diary,A_windowed,strict_herzog,NaN,50000,49605.0,...,0.00790,0.05604,False,NaN,NaN,C1 only,NaN,NaN,NaN,NaN
18045,pattern_decomposition,mutually_exclusive_patterns,healthy_ovulatory,full,full_diary,A_windowed,strict_herzog,NaN,50000,49605.0,...,0.00790,0.03440,False,NaN,NaN,C2 only,NaN,NaN,NaN,NaN
18046,pattern_decomposition,mutually_exclusive_patterns,healthy_ovulatory,full,full_diary,A_windowed,strict_herzog,NaN,50000,49605.0,...,0.00790,0.02218,False,NaN,NaN,C1+C2,NaN,NaN,NaN,NaN
18047,pattern_decomposition,mutually_exclusive_patterns,healthy_ovulatory,full,full_diary,A_windowed,strict_herzog,NaN,50000,49605.0,...,0.00790,0.00000,False,NaN,NaN,C3 only,NaN,NaN,NaN,NaN
18048,pattern_decomposition,mutually_exclusive_patterns,healthy_ovulatory,full,full_diary,A_windowed,strict_herzog,NaN,50000,49605.0,...,0.00790,0.00000,False,NaN,NaN,C3 plus C1/C2,NaN,NaN,NaN,NaN
18049,pattern_decomposition,mutually_exclusive_patterns,healthy_ovulatory,full,full_diary,A_windowed,strict_herzog,NaN,50000,49605.0,...,0.00790,0.87948,False,NaN,NaN,none,NaN,NaN,NaN,NaN
18200,pattern_decomposition,mutually_exclusive_patterns,population,full,full_diary,A_windowed,strict_herzog,NaN,50000,49587.0,...,0.00826,0.03724,False,NaN,NaN,C1 only,NaN,NaN,NaN,NaN
18201,pattern_decomposition,mutually_exclusive_patterns,population,full,full_diary,A_windowed,strict_herzog,NaN,50000,49587.0,...,0.00826,0.02110,False,NaN,NaN,C2 only,NaN,NaN,NaN,NaN
18202,pattern_decomposition,mutually_exclusive_patterns,population,full,full_diary,A_windowed,strict_herzog,NaN,50000,49587.0,...,0.00826,0.01128,False,NaN,NaN,C1+C2,NaN,NaN,NaN,NaN
18203,pattern_decomposition,mutually_exclusive_patterns,population,full,full_diary,A_windowed,strict_herzog,NaN,50000,49587.0,...,0.00826,0.24064,False,NaN,NaN,C3 only,NaN,NaN,NaN,NaN


| Cohort                         | CE definition   | Pattern category | Classifiable windows | False-positive windows | False-positive rate | Rate among all attempted | Indeterminate windows |
| ------------------------------ | --------------- | ---------------- | -------------------- | ---------------------- | ------------------- | ------------------------ | --------------------- |
| healthy ovulatory              | A_windowed      | C1 only          | 49,605               | 2,802                  | 5.6%                | 5.6%                     | 0.8%                  |
| healthy ovulatory              | A_windowed      | C1+C2            | 49,605               | 1,109                  | 2.2%                | 2.2%                     | 0.8%                  |
| healthy ovulatory              | A_windowed      | C2 only          | 49,605               | 1,720                  | 3.5%                | 3.4%                     | 0.8%                  |
| healthy ovulatory              | A_windowed      | C3 only          | 49,605               | 0                      | 0.0%                | 0.0%                     | 0.8%                  |
| healthy ovulatory              | A_windowed      | C3 plus C1/C2    | 49,605               | 0                      | 0.0%                | 0.0%                     | 0.8%                  |
| healthy ovulatory              | A_windowed      | none             | 49,605               | 43,974                 | 88.6%               | 87.9%                    | 0.8%                  |
| healthy ovulatory              | B_minimum_data  | C1 only          | 48,359               | 2,546                  | 5.3%                | 5.1%                     | 3.3%                  |
| healthy ovulatory              | B_minimum_data  | C1+C2            | 48,359               | 991                    | 2.0%                | 2.0%                     | 3.3%                  |
| healthy ovulatory              | B_minimum_data  | C2 only          | 48,359               | 1,456                  | 3.0%                | 2.9%                     | 3.3%                  |
| healthy ovulatory              | B_minimum_data  | C3 only          | 48,359               | 0                      | 0.0%                | 0.0%                     | 3.3%                  |
| healthy ovulatory              | B_minimum_data  | C3 plus C1/C2    | 48,359               | 0                      | 0.0%                | 0.0%                     | 3.3%                  |
| healthy ovulatory              | B_minimum_data  | none             | 48,359               | 43,366                 | 89.7%               | 86.7%                    | 3.3%                  |
| healthy ovulatory              | D_nb_regression | C1 only          | 48,359               | 951                    | 2.0%                | 1.9%                     | 3.3%                  |
| healthy ovulatory              | D_nb_regression | C1+C2            | 48,359               | 289                    | 0.6%                | 0.6%                     | 3.3%                  |
| healthy ovulatory              | D_nb_regression | C2 only          | 48,359               | 837                    | 1.7%                | 1.7%                     | 3.3%                  |
| healthy ovulatory              | D_nb_regression | C3 only          | 48,359               | 0                      | 0.0%                | 0.0%                     | 3.3%                  |
| healthy ovulatory              | D_nb_regression | C3 plus C1/C2    | 48,359               | 0                      | 0.0%                | 0.0%                     | 3.3%                  |
| healthy ovulatory              | D_nb_regression | none             | 48,359               | 46,282                 | 95.7%               | 92.6%                    | 3.3%                  |
| heterogeneous menstruating-age | A_windowed      | C1 only          | 49,587               | 1,862                  | 3.8%                | 3.7%                     | 0.8%                  |
| heterogeneous menstruating-age | A_windowed      | C1+C2            | 49,587               | 564                    | 1.1%                | 1.1%                     | 0.8%                  |
| heterogeneous menstruating-age | A_windowed      | C2 only          | 49,587               | 1,055                  | 2.1%                | 2.1%                     | 0.8%                  |
| heterogeneous menstruating-age | A_windowed      | C3 only          | 49,587               | 12,032                 | 24.3%               | 24.1%                    | 0.8%                  |
| heterogeneous menstruating-age | A_windowed      | C3 plus C1/C2    | 49,587               | 2,261                  | 4.6%                | 4.5%                     | 0.8%                  |
| heterogeneous menstruating-age | A_windowed      | none             | 49,587               | 31,813                 | 64.2%               | 63.6%                    | 0.8%                  |
| heterogeneous menstruating-age | B_minimum_data  | C1 only          | 48,333               | 1,652                  | 3.4%                | 3.3%                     | 3.3%                  |
| heterogeneous menstruating-age | B_minimum_data  | C1+C2            | 48,333               | 483                    | 1.0%                | 1.0%                     | 3.3%                  |
| heterogeneous menstruating-age | B_minimum_data  | C2 only          | 48,333               | 850                    | 1.8%                | 1.7%                     | 3.3%                  |
| heterogeneous menstruating-age | B_minimum_data  | C3 only          | 48,333               | 11,933                 | 24.7%               | 23.9%                    | 3.3%                  |
| heterogeneous menstruating-age | B_minimum_data  | C3 plus C1/C2    | 48,333               | 2,085                  | 4.3%                | 4.2%                     | 3.3%                  |
| heterogeneous menstruating-age | B_minimum_data  | none             | 48,333               | 31,330                 | 64.8%               | 62.7%                    | 3.3%                  |
| heterogeneous menstruating-age | D_nb_regression | C1 only          | 48,333               | 1,021                  | 2.1%                | 2.0%                     | 3.3%                  |
| heterogeneous menstruating-age | D_nb_regression | C1+C2            | 48,333               | 258                    | 0.5%                | 0.5%                     | 3.3%                  |
| heterogeneous menstruating-age | D_nb_regression | C2 only          | 48,333               | 784                    | 1.6%                | 1.6%                     | 3.3%                  |
| heterogeneous menstruating-age | D_nb_regression | C3 only          | 48,333               | 0                      | 0.0%                | 0.0%                     | 3.3%                  |
| heterogeneous menstruating-age | D_nb_regression | C3 plus C1/C2    | 48,333               | 0                      | 0.0%                | 0.0%                     | 3.3%                  |
| heterogeneous menstruating-age | D_nb_regression | none             | 48,333               | 46,270                 | 95.7%               | 92.5%                    | 3.3%                  |

**Table 7 caption.** Full-diary pattern decomposition and C3-exclusion sensitivity. C3 is evaluated only when ILP logic is applicable.

## Table 8. Negative-binomial dispersion sensitivity

**Why this table is included.** This table separates the full-diary stabilized-dispersion regression comparator from a full-window, window-only dispersion sensitivity.

**Code to call.**

In [10]:
nb_dispersion_sensitivity = summary_tables[
    (summary_tables.table_type == "window_false_positive")
    & (summary_tables.phase_mode == "strict_herzog")
    & (summary_tables.subset == "all")
    & (summary_tables.window_type == "full")
    & (summary_tables.definition.isin([
        "D_nb_regression_C1_or_C2", "D_nb_regression_window_alpha_C1_or_C2"
    ]))
].copy()
nb_dispersion_sensitivity


,table_type,subset,cohort,window_type,window_value,definition,phase_mode,assumption_based_historical,n_windows,n_classifiable,...,indeterminate_rate,positive_rate_all_attempted,unstable_denominator,interpretation_note,n_participants,pattern_category,indeterminate_reason,n_indeterminate,p_prevalence_ge_39_1,p_prevalence_ge_44_2
805,window_false_positive,all,healthy_ovulatory,full,full_diary,D_nb_regression_C1_or_C2,strict_herzog,False,50000,48359.0,...,0.03282,0.04154,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
831,window_false_positive,all,population,full,full_diary,D_nb_regression_C1_or_C2,strict_herzog,False,50000,48333.0,...,0.03334,0.04126,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
909,window_false_positive,all,healthy_ovulatory,full,full_diary,D_nb_regression_window_alpha_C1_or_C2,strict_herzog,False,50000,48359.0,...,0.03282,0.04154,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
935,window_false_positive,all,population,full,full_diary,D_nb_regression_window_alpha_C1_or_C2,strict_herzog,False,50000,48333.0,...,0.03334,0.04126,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


| Cohort                         | Observation window  | CE definition                                              | Classifiable windows | False-positive windows | False-positive rate (95% CI) | Indeterminate windows |
| ------------------------------ | ------------------- | ---------------------------------------------------------- | -------------------- | ---------------------- | ---------------------------- | --------------------- |
| healthy ovulatory              | 36-month full diary | Negative-binomial regression C1/C2                         | 48,359               | 2,077                  | 4.3% (4.1, 4.5)              | 3.3%                  |
| healthy ovulatory              | 36-month full diary | Negative-binomial regression C1/C2, window-only dispersion | 48,359               | 2,077                  | 4.3% (4.1, 4.5)              | 3.3%                  |
| heterogeneous menstruating-age | 36-month full diary | Negative-binomial regression C1/C2                         | 48,333               | 2,063                  | 4.3% (4.1, 4.5)              | 3.3%                  |
| heterogeneous menstruating-age | 36-month full diary | Negative-binomial regression C1/C2, window-only dispersion | 48,333               | 2,063                  | 4.3% (4.1, 4.5)              | 3.3%                  |

**Table 8 caption.** Negative-binomial apparent classification rates in full-diary windows using full-diary stabilized alpha and window-only alpha. Both use the same M/O model and Holm family.

## Table 9. Seizure-burden and cycle-regularity strata

**Why this table is included.** The requested strata diagnose where false positives concentrate. Seizure-frequency strata use observed full-diary seizure-days per month, and window-seizure-day strata use total seizure days within each analyzed window.

**Code to call.**

In [11]:
strata_rows = summary_tables[
    (summary_tables.table_type == "window_false_positive")
    & (summary_tables.phase_mode == "strict_herzog")
    & (summary_tables.definition.isin(["A_windowed_any", "A_windowed_C1_or_C2", "B_minimum_data_C1_or_C2", "D_nb_regression_C1_or_C2"]))
    & (
        summary_tables.subset.astype(str).str.startswith("seizure_frequency:")
        | summary_tables.subset.astype(str).str.startswith("cycle_regularity:")
        | summary_tables.subset.astype(str).str.startswith("window_seizure_days_")
    )
].copy()
strata_rows


,table_type,subset,cohort,window_type,window_value,definition,phase_mode,assumption_based_historical,n_windows,n_classifiable,...,indeterminate_rate,positive_rate_all_attempted,unstable_denominator,interpretation_note,n_participants,pattern_category,indeterminate_reason,n_indeterminate,p_prevalence_ge_39_1,p_prevalence_ge_44_2
4895,window_false_positive,window_seizure_days_0_to_3,healthy_ovulatory,calendar,1,A_windowed_any,strict_herzog,False,35302,23498.0,...,0.334372,0.332219,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4896,window_false_positive,window_seizure_days_0_to_3,healthy_ovulatory,calendar,3,A_windowed_any,strict_herzog,False,15576,10897.0,...,0.300398,0.356317,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4897,window_false_positive,window_seizure_days_0_to_3,healthy_ovulatory,calendar,4,A_windowed_any,strict_herzog,False,12373,8675.0,...,0.298877,0.357876,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4898,window_false_positive,window_seizure_days_0_to_3,healthy_ovulatory,calendar,6,A_windowed_any,strict_herzog,False,8860,6393.0,...,0.278442,0.376185,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4899,window_false_positive,window_seizure_days_0_to_3,healthy_ovulatory,calendar,9,A_windowed_any,strict_herzog,False,6195,4526.0,...,0.269411,0.373527,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16329,window_false_positive,cycle_regularity:SD cycle length >=4 days,population,calendar,36,D_nb_regression_C1_or_C2,strict_herzog,False,36317,0.0,...,1.000000,0.000000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16330,window_false_positive,cycle_regularity:SD cycle length >=4 days,population,cycle,3,D_nb_regression_C1_or_C2,strict_herzog,False,36317,0.0,...,1.000000,0.000000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16331,window_false_positive,cycle_regularity:SD cycle length >=4 days,population,cycle,6,D_nb_regression_C1_or_C2,strict_herzog,False,36317,0.0,...,1.000000,0.000000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16332,window_false_positive,cycle_regularity:SD cycle length >=4 days,population,cycle,12,D_nb_regression_C1_or_C2,strict_herzog,False,36317,0.0,...,1.000000,0.000000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


| Cohort                         | Stratum type               | Stratum                           | CE definition                           | Classifiable windows | False-positive windows | False-positive rate | Indeterminate windows |
| ------------------------------ | -------------------------- | --------------------------------- | --------------------------------------- | -------------------- | ---------------------- | ------------------- | --------------------- |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 3,108                | 1,473                  | 47.4%               | 23.2%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 3,678                | 1,534                  | 41.7%               | 9.1%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 3,731                | 1,461                  | 39.2%               | 7.8%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 3,831                | 1,324                  | 34.6%               | 5.4%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 3,903                | 1,155                  | 29.6%               | 3.6%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 3,932                | 985                    | 25.1%               | 2.9%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 3,969                | 789                    | 19.9%               | 2.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 3,985                | 656                    | 16.5%               | 1.6%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 4,008                | 493                    | 12.3%               | 1.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 3,619                | 1,497                  | 41.4%               | 10.6%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 3,836                | 1,348                  | 35.1%               | 5.2%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 3,930                | 1,042                  | 26.5%               | 2.9%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 4,008                | 493                    | 12.3%               | 1.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 3,108                | 1,473                  | 47.4%               | 23.2%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 3,678                | 1,534                  | 41.7%               | 9.1%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 3,731                | 1,461                  | 39.2%               | 7.8%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 3,831                | 1,324                  | 34.6%               | 5.4%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 3,903                | 1,155                  | 29.6%               | 3.6%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 3,932                | 985                    | 25.1%               | 2.9%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 3,969                | 789                    | 19.9%               | 2.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 3,985                | 656                    | 16.5%               | 1.6%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 4,008                | 493                    | 12.3%               | 1.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 3,619                | 1,497                  | 41.4%               | 10.6%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 3,836                | 1,348                  | 35.1%               | 5.2%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 3,930                | 1,042                  | 26.5%               | 2.9%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 4,008                | 493                    | 12.3%               | 1.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 3,045                | 1,113                  | 36.6%               | 24.8%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 3,304                | 1,027                  | 31.1%               | 18.4%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 3,553                | 979                    | 27.6%               | 12.2%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 3,658                | 850                    | 23.2%               | 9.6%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 3,777                | 685                    | 18.1%               | 6.7%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 3,828                | 567                    | 14.8%               | 5.4%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 3,889                | 426                    | 11.0%               | 3.9%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 3,278                | 1,054                  | 32.2%               | 19.0%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 3,624                | 880                    | 24.3%               | 10.5%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 3,889                | 426                    | 11.0%               | 3.9%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 3,889                | 172                    | 4.4%                | 3.9%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 10,948               | 5,193                  | 47.4%               | 23.6%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 12,991               | 5,493                  | 42.3%               | 9.4%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 13,277               | 5,143                  | 38.7%               | 7.4%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 13,645               | 4,758                  | 34.9%               | 4.8%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 13,849               | 4,097                  | 29.6%               | 3.4%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 13,972               | 3,529                  | 25.3%               | 2.6%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 14,103               | 2,731                  | 19.4%               | 1.6%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 14,168               | 2,291                  | 16.2%               | 1.2%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 14,225               | 1,646                  | 11.6%               | 0.8%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 12,943               | 5,490                  | 42.4%               | 9.7%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 13,555               | 4,771                  | 35.2%               | 5.5%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 13,955               | 3,672                  | 26.3%               | 2.7%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 14,225               | 1,646                  | 11.6%               | 0.8%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 10,948               | 5,193                  | 47.4%               | 23.6%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 12,991               | 5,493                  | 42.3%               | 9.4%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 13,277               | 5,143                  | 38.7%               | 7.4%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 13,645               | 4,758                  | 34.9%               | 4.8%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 13,849               | 4,097                  | 29.6%               | 3.4%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 13,972               | 3,529                  | 25.3%               | 2.6%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 14,103               | 2,731                  | 19.4%               | 1.6%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 14,168               | 2,291                  | 16.2%               | 1.2%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 14,225               | 1,646                  | 11.6%               | 0.8%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 12,943               | 5,490                  | 42.4%               | 9.7%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 13,555               | 4,771                  | 35.2%               | 5.5%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 13,955               | 3,672                  | 26.3%               | 2.7%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 14,225               | 1,646                  | 11.6%               | 0.8%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 10,699               | 3,826                  | 35.8%               | 25.4%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 11,771               | 3,743                  | 31.8%               | 17.9%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 12,551               | 3,410                  | 27.2%               | 12.5%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 12,964               | 2,993                  | 23.1%               | 9.6%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 13,387               | 2,353                  | 17.6%               | 6.6%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 13,639               | 2,016                  | 14.8%               | 4.9%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 13,872               | 1,458                  | 10.5%               | 3.3%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 11,656               | 3,774                  | 32.4%               | 18.7%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 12,886               | 3,113                  | 24.2%               | 10.1%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 13,872               | 1,458                  | 10.5%               | 3.3%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 13,872               | 593                    | 4.3%                | 3.3%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 24,130               | 11,434                 | 47.4%               | 23.7%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 28,652               | 11,841                 | 41.3%               | 9.4%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 29,294               | 11,261                 | 38.4%               | 7.3%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 30,057               | 10,056                 | 33.5%               | 4.9%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 30,579               | 8,941                  | 29.2%               | 3.3%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 30,827               | 7,663                  | 24.9%               | 2.5%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 31,087               | 6,044                  | 19.4%               | 1.7%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 31,228               | 4,992                  | 16.0%               | 1.2%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 31,372               | 3,492                  | 11.1%               | 0.8%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 28,507               | 11,712                 | 41.1%               | 9.8%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 30,036               | 10,348                 | 34.5%               | 5.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 30,783               | 7,666                  | 24.9%               | 2.6%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 31,372               | 3,492                  | 11.1%               | 0.8%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 24,130               | 11,434                 | 47.4%               | 23.7%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 28,652               | 11,841                 | 41.3%               | 9.4%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 29,294               | 11,261                 | 38.4%               | 7.3%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 30,057               | 10,056                 | 33.5%               | 4.9%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 30,579               | 8,941                  | 29.2%               | 3.3%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 30,827               | 7,663                  | 24.9%               | 2.5%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 31,087               | 6,044                  | 19.4%               | 1.7%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 31,228               | 4,992                  | 16.0%               | 1.2%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 31,372               | 3,492                  | 11.1%               | 0.8%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 28,507               | 11,712                 | 41.1%               | 9.8%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 30,036               | 10,348                 | 34.5%               | 5.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 30,783               | 7,666                  | 24.9%               | 2.6%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 31,372               | 3,492                  | 11.1%               | 0.8%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 23,883               | 8,498                  | 35.6%               | 24.5%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 26,065               | 8,035                  | 30.8%               | 17.6%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 27,701               | 7,490                  | 27.0%               | 12.4%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 28,646               | 6,551                  | 22.9%               | 9.4%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 29,559               | 5,303                  | 17.9%               | 6.5%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 30,068               | 4,400                  | 14.6%               | 4.9%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 30,598               | 3,109                  | 10.2%               | 3.2%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 103                  | 31                     | 30.1%               | 99.7%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 25,926               | 8,242                  | 31.8%               | 18.0%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 28,523               | 6,505                  | 22.8%               | 9.8%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 30,598               | 3,109                  | 10.2%               | 3.2%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 30,598               | 1,312                  | 4.3%                | 3.2%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 23,872               | 11,454                 | 48.0%               | 12.2%                 |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,010               | 11,236                 | 41.6%               | 0.6%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,124               | 10,326                 | 38.1%               | 0.2%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,170               | 8,873                  | 32.7%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,175               | 7,308                  | 26.9%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,175               | 5,962                  | 21.9%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,175               | 4,228                  | 15.6%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,175               | 3,106                  | 11.4%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,175               | 1,724                  | 6.3%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 26,952               | 11,232                 | 41.7%               | 0.8%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,166               | 9,168                  | 33.7%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,175               | 6,168                  | 22.7%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,175               | 1,724                  | 6.3%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 23,872               | 11,454                 | 48.0%               | 12.2%                 |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,010               | 11,236                 | 41.6%               | 0.6%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,124               | 10,326                 | 38.1%               | 0.2%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,170               | 8,873                  | 32.7%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,175               | 7,308                  | 26.9%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,175               | 5,962                  | 21.9%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,175               | 4,228                  | 15.6%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,175               | 3,106                  | 11.4%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,175               | 1,724                  | 6.3%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 26,952               | 11,232                 | 41.7%               | 0.8%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,166               | 9,168                  | 33.7%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,175               | 6,168                  | 22.7%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,175               | 1,724                  | 6.3%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 25,333               | 9,404                  | 37.1%               | 6.8%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 26,833               | 8,711                  | 32.5%               | 1.3%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 27,159               | 7,299                  | 26.9%               | 0.1%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 27,172               | 5,960                  | 21.9%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 27,175               | 4,228                  | 15.6%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 27,175               | 3,106                  | 11.4%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 27,175               | 1,724                  | 6.3%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 66                   | 19                     | 28.8%               | 99.8%                 |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 26,704               | 8,931                  | 33.4%               | 1.7%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 27,172               | 6,167                  | 22.7%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 27,175               | 1,724                  | 6.3%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 27,175               | 1,341                  | 4.9%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,065               | 4,539                  | 45.1%               | 0.7%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,134               | 3,560                  | 35.1%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,134               | 3,105                  | 30.6%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,134               | 2,368                  | 23.4%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,134               | 1,847                  | 18.2%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,134               | 1,373                  | 13.5%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,134               | 839                    | 8.3%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,134               | 571                    | 5.6%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,134               | 258                    | 2.5%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,134               | 3,455                  | 34.1%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,134               | 2,454                  | 24.2%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,134               | 1,414                  | 14.0%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,134               | 258                    | 2.5%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,065               | 4,539                  | 45.1%               | 0.7%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,134               | 3,560                  | 35.1%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,134               | 3,105                  | 30.6%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,134               | 2,368                  | 23.4%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,134               | 1,847                  | 18.2%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,134               | 1,373                  | 13.5%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,134               | 839                    | 8.3%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,134               | 571                    | 5.6%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,134               | 258                    | 2.5%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,134               | 3,455                  | 34.1%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,134               | 2,454                  | 24.2%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,134               | 1,414                  | 14.0%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,134               | 258                    | 2.5%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,132               | 3,103                  | 30.6%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,134               | 2,368                  | 23.4%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,134               | 1,847                  | 18.2%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,134               | 1,373                  | 13.5%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,134               | 839                    | 8.3%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,134               | 571                    | 5.6%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,134               | 258                    | 2.5%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 34                   | 10                     | 29.4%               | 99.7%                 |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,134               | 2,454                  | 24.2%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,134               | 1,414                  | 14.0%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,134               | 258                    | 2.5%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 10,134               | 360                    | 3.6%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 4,249                | 2,107                  | 49.6%               | 66.5%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 8,177                | 4,072                  | 49.8%               | 35.6%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 9,044                | 4,434                  | 49.0%               | 28.7%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 10,229               | 4,897                  | 47.9%               | 19.4%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 11,022               | 5,038                  | 45.7%               | 13.2%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 11,422               | 4,842                  | 42.4%               | 10.0%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 11,850               | 4,497                  | 37.9%               | 6.6%                  |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 12,072               | 4,262                  | 35.3%               | 4.9%                  |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 12,296               | 3,649                  | 29.7%               | 3.1%                  |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 7,983                | 4,012                  | 50.3%               | 37.1%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 10,127               | 4,845                  | 47.8%               | 20.2%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 11,359               | 4,798                  | 42.2%               | 10.5%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 12,296               | 3,649                  | 29.7%               | 3.1%                  |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 4,249                | 2,107                  | 49.6%               | 66.5%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 8,177                | 4,072                  | 49.8%               | 35.6%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 9,044                | 4,434                  | 49.0%               | 28.7%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 10,229               | 4,897                  | 47.9%               | 19.4%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 11,022               | 5,038                  | 45.7%               | 13.2%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 11,422               | 4,842                  | 42.4%               | 10.0%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 11,850               | 4,497                  | 37.9%               | 6.6%                  |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 12,072               | 4,262                  | 35.3%               | 4.9%                  |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 12,296               | 3,649                  | 29.7%               | 3.1%                  |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 7,983                | 4,012                  | 50.3%               | 37.1%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 10,127               | 4,845                  | 47.8%               | 20.2%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 11,359               | 4,798                  | 42.2%               | 10.5%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 12,296               | 3,649                  | 29.7%               | 3.1%                  |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 2,162                | 930                    | 43.0%               | 83.0%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 4,173                | 1,726                  | 41.4%               | 67.1%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 6,512                | 2,733                  | 42.0%               | 48.7%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 7,962                | 3,061                  | 38.4%               | 37.3%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 9,414                | 3,274                  | 34.8%               | 25.8%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 10,226               | 3,306                  | 32.3%               | 19.4%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 11,050               | 3,011                  | 27.2%               | 12.9%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 3                    | 2                      | 66.7%               | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 4,022                | 1,685                  | 41.9%               | 68.3%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 7,727                | 2,917                  | 37.8%               | 39.1%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 11,050               | 3,011                  | 27.2%               | 12.9%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 11,050               | 376                    | 3.4%                | 12.9%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 23,498               | 11,728                 | 49.9%               | 33.4%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 10,897               | 5,550                  | 50.9%               | 30.0%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 8,675                | 4,428                  | 51.0%               | 29.9%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 6,393                | 3,333                  | 52.1%               | 27.8%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 4,526                | 2,314                  | 51.1%               | 26.9%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 3,463                | 1,783                  | 51.5%               | 26.8%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 2,436                | 1,223                  | 50.2%               | 25.7%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 1,846                | 956                    | 51.8%               | 25.1%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 1,246                | 638                    | 51.2%               | 24.1%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 11,372               | 5,826                  | 51.2%               | 30.2%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 6,567                | 3,397                  | 51.7%               | 28.2%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 3,635                | 1,882                  | 51.8%               | 26.8%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 1,246                | 638                    | 51.2%               | 24.1%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 23,498               | 11,728                 | 49.9%               | 33.4%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 10,897               | 5,550                  | 50.9%               | 30.0%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 8,675                | 4,428                  | 51.0%               | 29.9%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 6,393                | 3,333                  | 52.1%               | 27.8%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 4,526                | 2,314                  | 51.1%               | 26.9%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 3,463                | 1,783                  | 51.5%               | 26.8%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 2,436                | 1,223                  | 50.2%               | 25.7%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 1,846                | 956                    | 51.8%               | 25.1%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 1,246                | 638                    | 51.2%               | 24.1%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 11,372               | 5,826                  | 51.2%               | 30.2%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 6,567                | 3,397                  | 51.7%               | 28.2%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 3,635                | 1,882                  | 51.8%               | 26.8%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 1,246                | 638                    | 51.2%               | 24.1%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 13,412               | 5,912                  | 44.1%               | 0.1%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 11,538               | 4,966                  | 43.0%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 9,419                | 3,970                  | 42.1%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 7,036                | 2,971                  | 42.2%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 5,178                | 2,250                  | 43.5%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 4,164                | 1,740                  | 41.8%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 2,894                | 1,174                  | 40.6%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 2,315                | 947                    | 40.9%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 1,624                | 674                    | 41.5%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 11,843               | 5,056                  | 42.7%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 7,422                | 3,106                  | 41.8%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 4,214                | 1,737                  | 41.2%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 1,624                | 674                    | 41.5%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 13,412               | 5,912                  | 44.1%               | 0.1%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 11,538               | 4,966                  | 43.0%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 9,419                | 3,970                  | 42.1%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 7,036                | 2,971                  | 42.2%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 5,178                | 2,250                  | 43.5%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 4,164                | 1,740                  | 41.8%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 2,894                | 1,174                  | 40.6%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 2,315                | 947                    | 40.9%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 1,624                | 674                    | 41.5%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 11,843               | 5,056                  | 42.7%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 7,422                | 3,106                  | 41.8%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 4,214                | 1,737                  | 41.2%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 1,624                | 674                    | 41.5%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 9,419                | 3,970                  | 42.1%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 7,036                | 2,971                  | 42.2%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 5,178                | 2,250                  | 43.5%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 4,164                | 1,740                  | 41.8%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 2,894                | 1,174                  | 40.6%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 2,315                | 947                    | 40.9%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 1,624                | 674                    | 41.5%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 15                   | 6                      | 40.0%               | 99.9%                 |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 7,422                | 3,106                  | 41.8%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 4,214                | 1,737                  | 41.2%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 1,624                | 674                    | 41.5%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 1,624                | 17                     | 1.0%                | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 1,276                | 460                    | 36.1%               | 0.2%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 18,612               | 7,054                  | 37.9%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 16,361               | 6,011                  | 36.7%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 11,197               | 3,802                  | 34.0%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 8,763                | 3,033                  | 34.6%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 7,258                | 2,457                  | 33.9%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 5,308                | 1,820                  | 34.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 4,162                | 1,415                  | 34.0%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 2,948                | 991                    | 33.6%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 18,198               | 6,748                  | 37.1%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 11,624               | 4,061                  | 34.9%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 7,457                | 2,469                  | 33.1%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 2,948                | 991                    | 33.6%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 1,276                | 460                    | 36.1%               | 0.2%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 18,612               | 7,054                  | 37.9%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 16,361               | 6,011                  | 36.7%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 11,197               | 3,802                  | 34.0%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 8,763                | 3,033                  | 34.6%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 7,258                | 2,457                  | 33.9%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 5,308                | 1,820                  | 34.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 4,162                | 1,415                  | 34.0%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 2,948                | 991                    | 33.6%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 18,198               | 6,748                  | 37.1%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 11,624               | 4,061                  | 34.9%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 7,457                | 2,469                  | 33.1%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 2,948                | 991                    | 33.6%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 16,361               | 6,011                  | 36.7%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 11,197               | 3,802                  | 34.0%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 8,763                | 3,033                  | 34.6%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 7,258                | 2,457                  | 33.9%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 5,308                | 1,820                  | 34.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 4,162                | 1,415                  | 34.0%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 2,948                | 991                    | 33.6%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 46                   | 15                     | 32.6%               | 99.7%                 |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 11,624               | 4,061                  | 34.9%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 7,457                | 2,469                  | 33.1%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 2,948                | 991                    | 33.6%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 2,948                | 97                     | 3.3%                | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 4,274                | 1,298                  | 30.4%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 11,847               | 3,456                  | 29.2%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 22,907               | 6,032                  | 26.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 29,864               | 6,596                  | 22.1%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 33,846               | 6,197                  | 18.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 38,521               | 5,347                  | 13.9%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 41,058               | 4,621                  | 11.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 43,787               | 3,328                  | 7.6%                | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 3,656                | 1,069                  | 29.2%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 21,814               | 5,903                  | 27.1%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 33,362               | 6,292                  | 18.9%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 43,787               | 3,328                  | 7.6%                | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 4,274                | 1,298                  | 30.4%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 11,847               | 3,456                  | 29.2%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 22,907               | 6,032                  | 26.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 29,864               | 6,596                  | 22.1%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 33,846               | 6,197                  | 18.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 38,521               | 5,347                  | 13.9%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 41,058               | 4,621                  | 11.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 43,787               | 3,328                  | 7.6%                | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 3,656                | 1,069                  | 29.2%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 21,814               | 5,903                  | 27.1%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 33,362               | 6,292                  | 18.9%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 43,787               | 3,328                  | 7.6%                | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 11,847               | 3,456                  | 29.2%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 22,907               | 6,032                  | 26.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 29,864               | 6,596                  | 22.1%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 33,846               | 6,197                  | 18.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 38,521               | 5,347                  | 13.9%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 41,058               | 4,621                  | 11.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 43,787               | 3,328                  | 7.6%                | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 42                   | 10                     | 23.8%               | 98.9%                 |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 21,814               | 5,903                  | 27.1%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 33,362               | 6,292                  | 18.9%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 43,787               | 3,328                  | 7.6%                | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 43,787               | 1,963                  | 4.5%                | 0.0%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 1,997                | 991                    | 49.6%               | 22.6%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 2,338                | 957                    | 40.9%               | 9.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 2,393                | 938                    | 39.2%               | 7.2%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 2,455                | 856                    | 34.9%               | 4.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 2,512                | 773                    | 30.8%               | 2.6%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 2,522                | 635                    | 25.2%               | 2.2%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 2,545                | 514                    | 20.2%               | 1.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 2,555                | 422                    | 16.5%               | 0.9%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 2,565                | 285                    | 11.1%               | 0.5%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 2,335                | 936                    | 40.1%               | 9.5%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 2,435                | 900                    | 37.0%               | 5.6%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 2,518                | 636                    | 25.3%               | 2.4%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 2,565                | 285                    | 11.1%               | 0.5%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 1,997                | 1,024                  | 51.3%               | 22.6%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 2,338                | 1,083                  | 46.3%               | 9.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 2,393                | 1,082                  | 45.2%               | 7.2%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 2,455                | 1,089                  | 44.4%               | 4.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 2,512                | 1,071                  | 42.6%               | 2.6%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 2,522                | 993                    | 39.4%               | 2.2%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 2,545                | 979                    | 38.5%               | 1.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 2,555                | 980                    | 38.4%               | 0.9%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 2,565                | 977                    | 38.1%               | 0.5%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 2,335                | 1,064                  | 45.6%               | 9.5%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 2,435                | 1,097                  | 45.1%               | 5.6%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 2,518                | 970                    | 38.5%               | 2.4%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 2,565                | 977                    | 38.1%               | 0.5%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 1,923                | 697                    | 36.2%               | 25.4%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 2,111                | 687                    | 32.5%               | 18.1%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 2,278                | 644                    | 28.3%               | 11.7%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 2,342                | 536                    | 22.9%               | 9.2%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 2,428                | 453                    | 18.7%               | 5.9%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 2,461                | 366                    | 14.9%               | 4.6%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 2,501                | 248                    | 9.9%                | 3.0%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 1                    | 1                      | 100.0%              | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 2,103                | 739                    | 35.1%               | 18.5%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 2,319                | 530                    | 22.9%               | 10.1%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 2,501                | 248                    | 9.9%                | 3.0%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 2,501                | 97                     | 3.9%                | 3.0%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 8,450                | 4,152                  | 49.1%               | 23.9%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 10,036               | 4,151                  | 41.4%               | 9.6%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 10,251               | 4,017                  | 39.2%               | 7.7%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 10,540               | 3,653                  | 34.7%               | 5.1%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 10,694               | 3,144                  | 29.4%               | 3.7%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 10,813               | 2,684                  | 24.8%               | 2.6%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 10,905               | 2,155                  | 19.8%               | 1.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 10,959               | 1,700                  | 15.5%               | 1.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 11,000               | 1,234                  | 11.2%               | 0.9%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 9,968                | 4,157                  | 41.7%               | 10.2%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 10,514               | 3,678                  | 35.0%               | 5.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 10,807               | 2,708                  | 25.1%               | 2.7%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 11,000               | 1,234                  | 11.2%               | 0.9%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 8,450                | 4,361                  | 51.6%               | 23.9%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 10,036               | 4,731                  | 47.1%               | 9.6%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 10,251               | 4,791                  | 46.7%               | 7.7%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 10,540               | 4,610                  | 43.7%               | 5.1%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 10,694               | 4,491                  | 42.0%               | 3.7%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 10,813               | 4,324                  | 40.0%               | 2.6%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 10,905               | 4,217                  | 38.7%               | 1.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 10,959               | 4,196                  | 38.3%               | 1.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 11,000               | 4,232                  | 38.5%               | 0.9%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 9,968                | 4,803                  | 48.2%               | 10.2%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 10,514               | 4,646                  | 44.2%               | 5.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 10,807               | 4,296                  | 39.8%               | 2.7%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 11,000               | 4,232                  | 38.5%               | 0.9%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 8,249                | 2,979                  | 36.1%               | 25.7%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 9,098                | 2,929                  | 32.2%               | 18.1%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 9,744                | 2,625                  | 26.9%               | 12.2%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 10,059               | 2,293                  | 22.8%               | 9.4%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 10,389               | 1,888                  | 18.2%               | 6.4%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 10,552               | 1,486                  | 14.1%               | 5.0%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 10,741               | 1,099                  | 10.2%               | 3.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 8                    | 5                      | 62.5%               | 99.9%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 9,027                | 2,877                  | 31.9%               | 18.7%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 10,011               | 2,299                  | 23.0%               | 9.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 10,741               | 1,099                  | 10.2%               | 3.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 10,741               | 472                    | 4.4%                | 3.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 27,525               | 13,167                 | 47.8%               | 24.2%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 32,820               | 13,989                 | 42.6%               | 9.6%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 33,657               | 13,044                 | 38.8%               | 7.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 34,458               | 11,988                 | 34.8%               | 5.1%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 35,100               | 10,252                 | 29.2%               | 3.4%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 35,384               | 9,033                  | 25.5%               | 2.6%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 35,689               | 7,194                  | 20.2%               | 1.7%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 35,859               | 5,868                  | 16.4%               | 1.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 36,022               | 4,223                  | 11.7%               | 0.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 32,903               | 13,583                 | 41.3%               | 9.4%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 34,528               | 11,734                 | 34.0%               | 4.9%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 35,395               | 8,861                  | 25.0%               | 2.5%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 36,022               | 4,223                  | 11.7%               | 0.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 27,525               | 14,441                 | 52.5%               | 24.2%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 32,820               | 17,223                 | 52.5%               | 9.6%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 33,657               | 17,038                 | 50.6%               | 7.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 34,458               | 16,661                 | 48.4%               | 5.1%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 35,100               | 15,777                 | 44.9%               | 3.4%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 35,384               | 15,142                 | 42.8%               | 2.6%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 35,689               | 14,119                 | 39.6%               | 1.7%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 35,859               | 13,449                 | 37.5%               | 1.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 36,022               | 12,565                 | 34.9%               | 0.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 32,903               | 16,977                 | 51.6%               | 9.4%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 34,528               | 16,425                 | 47.6%               | 4.9%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 35,395               | 14,811                 | 41.8%               | 2.5%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 36,022               | 12,565                 | 34.9%               | 0.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 27,194               | 9,773                  | 35.9%               | 25.1%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 29,954               | 9,650                  | 32.2%               | 17.5%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 31,881               | 8,581                  | 26.9%               | 12.2%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 32,877               | 7,733                  | 23.5%               | 9.5%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 33,940               | 6,281                  | 18.5%               | 6.5%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 34,501               | 5,157                  | 14.9%               | 5.0%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 35,091               | 3,723                  | 10.6%               | 3.4%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 1,578                | 625                    | 39.6%               | 95.7%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 29,985               | 9,361                  | 31.2%               | 17.4%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 32,919               | 7,558                  | 23.0%               | 9.4%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 35,091               | 3,723                  | 10.6%               | 3.4%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 35,091               | 1,494                  | 4.3%                | 3.4%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 23,713               | 11,550                 | 48.7%               | 12.4%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 26,880               | 11,304                 | 42.1%               | 0.7%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,017               | 10,413                 | 38.5%               | 0.2%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,064               | 9,032                  | 33.4%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,072               | 7,241                  | 26.7%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,072               | 6,041                  | 22.3%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,072               | 4,328                  | 16.0%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,072               | 3,089                  | 11.4%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,072               | 1,739                  | 6.4%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 26,859               | 11,028                 | 41.1%               | 0.8%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,062               | 8,893                  | 32.9%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,072               | 5,944                  | 22.0%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,072               | 1,739                  | 6.4%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 23,713               | 12,517                 | 52.8%               | 12.4%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 26,880               | 13,745                 | 51.1%               | 0.7%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,017               | 13,397                 | 49.6%               | 0.2%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,064               | 12,576                 | 46.5%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,072               | 11,520                 | 42.6%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,072               | 10,859                 | 40.1%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,072               | 9,970                  | 36.8%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,072               | 9,493                  | 35.1%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,072               | 8,918                  | 32.9%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 26,859               | 13,589                 | 50.6%               | 0.8%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,062               | 12,438                 | 46.0%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,072               | 10,655                 | 39.4%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,072               | 8,918                  | 32.9%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 25,141               | 9,467                  | 37.7%               | 7.1%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 26,742               | 8,856                  | 33.1%               | 1.2%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 27,052               | 7,233                  | 26.7%               | 0.1%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 27,072               | 6,041                  | 22.3%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 27,072               | 4,328                  | 16.0%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 27,072               | 3,089                  | 11.4%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 27,072               | 1,739                  | 6.4%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 1,054                | 424                    | 40.2%               | 96.1%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 26,644               | 8,680                  | 32.6%               | 1.6%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 27,071               | 5,943                  | 22.0%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 27,072               | 1,739                  | 6.4%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 27,072               | 1,320                  | 4.9%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 9,969                | 4,599                  | 46.1%               | 1.5%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,123               | 3,555                  | 35.1%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,124               | 3,080                  | 30.4%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,124               | 2,539                  | 25.1%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,124               | 1,895                  | 18.7%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,124               | 1,373                  | 13.6%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,124               | 879                    | 8.7%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,124               | 559                    | 5.5%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,124               | 296                    | 2.9%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,124               | 3,455                  | 34.1%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,124               | 2,460                  | 24.3%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,124               | 1,399                  | 13.8%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,124               | 296                    | 2.9%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 9,969                | 4,981                  | 50.0%               | 1.5%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,123               | 4,486                  | 44.3%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,124               | 4,253                  | 42.0%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,124               | 3,876                  | 38.3%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,124               | 3,490                  | 34.5%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,124               | 3,154                  | 31.2%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,124               | 2,913                  | 28.8%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,124               | 2,749                  | 27.2%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,124               | 2,639                  | 26.1%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,124               | 4,418                  | 43.6%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,124               | 3,784                  | 37.4%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,124               | 3,097                  | 30.6%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,124               | 2,639                  | 26.1%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,122               | 3,079                  | 30.4%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,124               | 2,539                  | 25.1%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,124               | 1,895                  | 18.7%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,124               | 1,373                  | 13.6%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,124               | 879                    | 8.7%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,124               | 559                    | 5.5%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,124               | 296                    | 2.9%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 405                  | 137                    | 33.8%               | 96.0%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,124               | 2,460                  | 24.3%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,124               | 1,399                  | 13.8%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,124               | 296                    | 2.9%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 10,124               | 372                    | 3.7%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 4,290                | 2,161                  | 50.4%               | 66.5%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 8,191                | 4,238                  | 51.7%               | 36.0%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 9,160                | 4,506                  | 49.2%               | 28.5%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 10,265               | 4,926                  | 48.0%               | 19.8%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 11,110               | 5,033                  | 45.3%               | 13.2%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 11,523               | 4,938                  | 42.9%               | 10.0%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 11,943               | 4,656                  | 39.0%               | 6.7%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 12,177               | 4,342                  | 35.7%               | 4.9%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 12,391               | 3,707                  | 29.9%               | 3.2%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 8,223                | 4,193                  | 51.0%               | 35.8%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 10,291               | 4,959                  | 48.2%               | 19.6%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 11,524               | 4,862                  | 42.2%               | 10.0%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 12,391               | 3,707                  | 29.9%               | 3.2%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 4,290                | 2,328                  | 54.3%               | 66.5%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 8,191                | 4,806                  | 58.7%               | 36.0%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 9,160                | 5,261                  | 57.4%               | 28.5%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 10,265               | 5,908                  | 57.6%               | 19.8%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 11,110               | 6,329                  | 57.0%               | 13.2%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 11,523               | 6,446                  | 55.9%               | 10.0%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 11,943               | 6,432                  | 53.9%               | 6.7%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 12,177               | 6,383                  | 52.4%               | 4.9%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 12,391               | 6,217                  | 50.2%               | 3.2%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 8,223                | 4,837                  | 58.8%               | 35.8%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 10,291               | 5,946                  | 57.8%               | 19.6%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 11,524               | 6,325                  | 54.9%               | 10.0%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 12,391               | 6,217                  | 50.2%               | 3.2%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 2,103                | 903                    | 42.9%               | 83.6%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 4,297                | 1,871                  | 43.5%               | 66.4%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 6,727                | 2,722                  | 40.5%               | 47.5%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 8,082                | 3,148                  | 39.0%               | 36.9%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 9,561                | 3,415                  | 35.7%               | 25.3%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 10,318               | 3,361                  | 32.6%               | 19.4%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 11,137               | 3,035                  | 27.3%               | 13.0%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 128                  | 70                     | 54.7%               | 99.0%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 4,347                | 1,837                  | 42.3%               | 66.0%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 8,054                | 3,045                  | 37.8%               | 37.1%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 11,137               | 3,035                  | 27.3%               | 13.0%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 11,137               | 371                    | 3.3%                | 13.0%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 23,451               | 11,906                 | 50.8%               | 33.7%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 10,824               | 5,653                  | 52.2%               | 30.7%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 8,935                | 4,550                  | 50.9%               | 29.3%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 6,290                | 3,231                  | 51.4%               | 28.8%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 4,403                | 2,319                  | 52.7%               | 27.8%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 3,441                | 1,790                  | 52.0%               | 27.1%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 2,382                | 1,241                  | 52.1%               | 26.5%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 1,859                | 981                    | 52.8%               | 25.2%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 1,254                | 672                    | 53.6%               | 24.8%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 11,090               | 5,718                  | 51.6%               | 30.2%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 6,362                | 3,335                  | 52.4%               | 28.4%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 3,471                | 1,818                  | 52.4%               | 26.9%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 1,254                | 672                    | 53.6%               | 24.8%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 23,451               | 12,881                 | 54.9%               | 33.7%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 10,824               | 6,398                  | 59.1%               | 30.7%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 8,935                | 5,259                  | 58.9%               | 29.3%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 6,290                | 3,729                  | 59.3%               | 28.8%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 4,403                | 2,702                  | 61.4%               | 27.8%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 3,441                | 2,089                  | 60.7%               | 27.1%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 2,382                | 1,451                  | 60.9%               | 26.5%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 1,859                | 1,148                  | 61.8%               | 25.2%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 1,254                | 771                    | 61.5%               | 24.8%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 11,090               | 6,499                  | 58.6%               | 30.2%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 6,362                | 3,831                  | 60.2%               | 28.4%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 3,471                | 2,100                  | 60.5%               | 26.9%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 1,254                | 771                    | 61.5%               | 24.8%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 13,270               | 5,904                  | 44.5%               | 0.6%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 11,542               | 5,050                  | 43.8%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 9,082                | 3,915                  | 43.1%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 7,152                | 3,057                  | 42.7%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 5,404                | 2,294                  | 42.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 4,172                | 1,765                  | 42.3%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 2,899                | 1,254                  | 43.3%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 2,243                | 929                    | 41.4%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 1,567                | 615                    | 39.2%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 11,646               | 4,875                  | 41.9%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 7,250                | 3,002                  | 41.4%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 4,231                | 1,749                  | 41.3%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 1,567                | 615                    | 39.2%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 13,270               | 6,383                  | 48.1%               | 0.6%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 11,542               | 6,133                  | 53.1%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 9,082                | 4,924                  | 54.2%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 7,152                | 3,940                  | 55.1%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 5,404                | 3,046                  | 56.4%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 4,172                | 2,343                  | 56.2%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 2,899                | 1,687                  | 58.2%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 2,243                | 1,289                  | 57.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 1,567                | 885                    | 56.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 11,646               | 6,028                  | 51.8%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 7,250                | 3,864                  | 53.3%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 4,231                | 2,324                  | 54.9%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 1,567                | 885                    | 56.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 9,082                | 3,915                  | 43.1%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 7,152                | 3,057                  | 42.7%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 5,404                | 2,294                  | 42.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 4,172                | 1,765                  | 42.3%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 2,899                | 1,254                  | 43.3%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 2,243                | 929                    | 41.4%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 1,567                | 615                    | 39.2%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 381                  | 186                    | 48.8%               | 96.7%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 7,250                | 3,002                  | 41.4%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 4,231                | 1,749                  | 41.3%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 1,567                | 615                    | 39.2%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 1,567                | 12                     | 0.8%                | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 1,251                | 500                    | 40.0%               | 0.5%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 18,544               | 7,119                  | 38.4%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 16,554               | 6,038                  | 36.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 11,113               | 3,865                  | 34.8%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 8,544                | 2,901                  | 34.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 7,261                | 2,494                  | 34.3%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 5,428                | 1,839                  | 33.9%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 4,227                | 1,479                  | 35.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 3,005                | 1,020                  | 33.9%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 17,820               | 6,663                  | 37.4%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 11,265               | 3,820                  | 33.9%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 7,258                | 2,391                  | 32.9%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 3,005                | 1,020                  | 33.9%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 1,251                | 562                    | 44.9%               | 0.5%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 18,544               | 8,865                  | 47.8%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 16,554               | 7,882                  | 47.6%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 11,113               | 5,275                  | 47.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 8,544                | 4,142                  | 48.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 7,261                | 3,676                  | 50.6%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 5,428                | 2,744                  | 50.6%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 4,227                | 2,213                  | 52.4%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 3,005                | 1,579                  | 52.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 17,820               | 8,356                  | 46.9%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 11,265               | 5,291                  | 47.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 7,258                | 3,550                  | 48.9%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 3,005                | 1,579                  | 52.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 16,554               | 6,038                  | 36.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 11,113               | 3,865                  | 34.8%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 8,544                | 2,901                  | 34.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 7,261                | 2,494                  | 34.3%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 5,428                | 1,839                  | 33.9%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 4,227                | 1,479                  | 35.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 3,005                | 1,020                  | 33.9%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 620                  | 250                    | 40.3%               | 96.5%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 11,265               | 3,820                  | 33.9%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 7,258                | 2,391                  | 32.9%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 3,005                | 1,020                  | 33.9%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 3,005                | 98                     | 3.3%                | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 4,284                | 1,275                  | 29.8%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 11,730               | 3,496                  | 29.8%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 22,898               | 6,344                  | 27.7%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 29,955               | 6,655                  | 22.2%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 33,845               | 6,303                  | 18.6%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 38,430               | 5,529                  | 14.4%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 41,044               | 4,601                  | 11.2%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 43,761               | 3,435                  | 7.8%                | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 4,650                | 1,420                  | 30.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 22,600               | 6,155                  | 27.2%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 33,760               | 6,247                  | 18.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 43,761               | 3,435                  | 7.8%                | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 4,284                | 1,641                  | 38.3%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 11,730               | 4,846                  | 41.3%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 22,898               | 9,416                  | 41.1%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 29,955               | 11,449                 | 38.2%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 33,845               | 12,351                 | 36.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 38,430               | 13,433                 | 35.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 41,044               | 13,975                 | 34.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 43,761               | 14,539                 | 33.2%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 4,650                | 1,961                  | 42.2%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 22,600               | 9,182                  | 40.6%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 33,760               | 12,103                 | 35.9%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 43,761               | 14,539                 | 33.2%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 11,730               | 3,496                  | 29.8%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 22,898               | 6,344                  | 27.7%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 29,955               | 6,655                  | 22.2%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 33,845               | 6,303                  | 18.6%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 38,430               | 5,529                  | 14.4%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 41,044               | 4,601                  | 11.2%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 43,761               | 3,435                  | 7.8%                | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 586                  | 195                    | 33.3%               | 87.4%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 22,600               | 6,155                  | 27.2%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 33,760               | 6,247                  | 18.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 43,761               | 3,435                  | 7.8%                | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 43,761               | 1,953                  | 4.5%                | 0.0%                  |

**Table 9 caption.** Apparent classification rates by observed seizure burden and cycle regularity strata. These strata identify where null positives are concentrated.

## Table 10. Assumption-based historical definitions

**Why this table is included.** Historical rules are exploratory operationalizations rather than literal replications, so they are flagged separately. This table keeps them out of the core endpoint table while still showing their null false-positive behavior.

**Code to call.**

In [12]:
historical_rows = summary_tables[
    (summary_tables.table_type == "window_false_positive")
    & (summary_tables.phase_mode == "strict_herzog")
    & (summary_tables.subset == "all")
    & (summary_tables.window_type.isin(["calendar", "full"]))
    & (summary_tables.definition.isin([
        "H1_newmark_penry_any", "H1_newmark_penry_66_7_any",
        "H2_duncan1993_any", "H3_herzog1997_twofold_any",
        "H4_reddy2007_any_phase2x_any"
    ]))
].copy()
historical_rows


,table_type,subset,cohort,window_type,window_value,definition,phase_mode,assumption_based_historical,n_windows,n_classifiable,...,indeterminate_rate,positive_rate_all_attempted,unstable_denominator,interpretation_note,n_participants,pattern_category,indeterminate_reason,n_indeterminate,p_prevalence_ge_39_1,p_prevalence_ge_44_2
949,window_false_positive,all,healthy_ovulatory,calendar,1,H1_newmark_penry_any,strict_herzog,True,50000,38214.0,...,0.23572,0.11912,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
950,window_false_positive,all,healthy_ovulatory,calendar,3,H1_newmark_penry_any,strict_herzog,True,50000,45321.0,...,0.09358,0.08202,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
951,window_false_positive,all,healthy_ovulatory,calendar,4,H1_newmark_penry_any,strict_herzog,True,50000,46302.0,...,0.07396,0.06718,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
952,window_false_positive,all,healthy_ovulatory,calendar,6,H1_newmark_penry_any,strict_herzog,True,50000,47533.0,...,0.04934,0.04862,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
953,window_false_positive,all,healthy_ovulatory,calendar,9,H1_newmark_penry_any,strict_herzog,True,50000,48331.0,...,0.03338,0.03426,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1188,window_false_positive,all,population,calendar,12,H4_reddy2007_any_phase2x_any,strict_herzog,True,50000,48719.0,...,0.02562,0.36076,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1189,window_false_positive,all,population,calendar,18,H4_reddy2007_any_phase2x_any,strict_herzog,True,50000,49139.0,...,0.01722,0.25908,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1190,window_false_positive,all,population,calendar,24,H4_reddy2007_any_phase2x_any,strict_herzog,True,50000,49373.0,...,0.01254,0.19794,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1191,window_false_positive,all,population,calendar,36,H4_reddy2007_any_phase2x_any,strict_herzog,True,50000,49587.0,...,0.00826,0.13350,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


| Cohort                         | Observation window  | CE definition                        | Classifiable windows | False-positive windows | False-positive rate (95% CI) | Indeterminate windows | Assumption-based historical rule |
| ------------------------------ | ------------------- | ------------------------------------ | -------------------- | ---------------------- | ---------------------------- | --------------------- | -------------------------------- |
| healthy ovulatory              | 3 months            | Newmark-Penry two-thirds sensitivity | 45,321               | 1,893                  | 4.2% (4.0, 4.4)              | 9.4%                  | Yes                              |
| healthy ovulatory              | 3 months            | Newmark-Penry perimenstrual rule     | 45,321               | 4,101                  | 9.0% (8.8, 9.3)              | 9.4%                  | Yes                              |
| healthy ovulatory              | 3 months            | Duncan 1993 ten-day rule             | 45,321               | 3,366                  | 7.4% (7.2, 7.7)              | 9.4%                  | Yes                              |
| healthy ovulatory              | 3 months            | Herzog 1997 twofold rule             | 45,321               | 16,914                 | 37.3% (36.9, 37.8)           | 9.4%                  | Yes                              |
| healthy ovulatory              | 3 months            | Reddy 2007 any-phase twofold rule    | 45,321               | 34,654                 | 76.5% (76.1, 76.9)           | 9.4%                  | Yes                              |
| healthy ovulatory              | 36-month full diary | Newmark-Penry two-thirds sensitivity | 49,605               | 161                    | 0.3% (0.3, 0.4)              | 0.8%                  | Yes                              |
| healthy ovulatory              | 36-month full diary | Newmark-Penry perimenstrual rule     | 49,605               | 428                    | 0.9% (0.8, 0.9)              | 0.8%                  | Yes                              |
| healthy ovulatory              | 36-month full diary | Duncan 1993 ten-day rule             | 49,605               | 364                    | 0.7% (0.7, 0.8)              | 0.8%                  | Yes                              |
| healthy ovulatory              | 36-month full diary | Herzog 1997 twofold rule             | 49,605               | 3,741                  | 7.5% (7.3, 7.8)              | 0.8%                  | Yes                              |
| healthy ovulatory              | 36-month full diary | Reddy 2007 any-phase twofold rule    | 49,605               | 6,730                  | 13.6% (13.3, 13.9)           | 0.8%                  | Yes                              |
| heterogeneous menstruating-age | 3 months            | Newmark-Penry two-thirds sensitivity | 45,198               | 1,831                  | 4.1% (3.9, 4.2)              | 9.6%                  | Yes                              |
| heterogeneous menstruating-age | 3 months            | Newmark-Penry perimenstrual rule     | 45,198               | 3,886                  | 8.6% (8.3, 8.9)              | 9.6%                  | Yes                              |
| heterogeneous menstruating-age | 3 months            | Duncan 1993 ten-day rule             | 45,198               | 3,108                  | 6.9% (6.6, 7.1)              | 9.6%                  | Yes                              |
| heterogeneous menstruating-age | 3 months            | Herzog 1997 twofold rule             | 45,194               | 21,247                 | 47.0% (46.6, 47.5)           | 9.6%                  | Yes                              |
| heterogeneous menstruating-age | 3 months            | Reddy 2007 any-phase twofold rule    | 45,195               | 34,562                 | 76.5% (76.1, 76.9)           | 9.6%                  | Yes                              |
| heterogeneous menstruating-age | 36-month full diary | Newmark-Penry two-thirds sensitivity | 49,587               | 169                    | 0.3% (0.3, 0.4)              | 0.8%                  | Yes                              |
| heterogeneous menstruating-age | 36-month full diary | Newmark-Penry perimenstrual rule     | 49,587               | 407                    | 0.8% (0.7, 0.9)              | 0.8%                  | Yes                              |
| heterogeneous menstruating-age | 36-month full diary | Duncan 1993 ten-day rule             | 49,587               | 325                    | 0.7% (0.6, 0.7)              | 0.8%                  | Yes                              |
| heterogeneous menstruating-age | 36-month full diary | Herzog 1997 twofold rule             | 49,587               | 14,714                 | 29.7% (29.3, 30.1)           | 0.8%                  | Yes                              |
| heterogeneous menstruating-age | 36-month full diary | Reddy 2007 any-phase twofold rule    | 49,587               | 6,675                  | 13.5% (13.2, 13.8)           | 0.8%                  | Yes                              |

**Table 10 caption.** Apparent classification rates for exploratory historical definitions. These rows are deliberately labeled as assumption-based and should not be interpreted as literal historical replications.

## Table 11. Output manifest

**Why this table is included.** The manifest is machine-readable provenance: it lists every analysis artifact, size, checksum, and the assumptions that were not directly derivable from simulator outputs.

**Code to call.**

In [13]:
manifest_files = pd.DataFrame(manifest["files"])
manifest_files.assign(size_mb=manifest_files["bytes"] / 1_000_000)[["path", "size_mb", "sha256"]]


,path,size_mb,sha256
0,outputs/random_start_full_v14_waveform_recalib...,6.977710,3cde064a6366a826b501a9bc8666de4d310c622a1eb489...
1,outputs/random_start_full_v14_waveform_recalib...,0.021294,b93c78078e5f9b2e0259dc790e1be9168497b10eb12c05...
2,outputs/random_start_full_v14_waveform_recalib...,0.134186,c5dc23f0d5dce7b1336902009360c392507bb1fcc25d99...
3,outputs/random_start_full_v14_waveform_recalib...,0.080101,1e25b445218c01c92188f93f5beab96013fdc0b09ce22c...
4,outputs/random_start_full_v14_waveform_recalib...,0.018513,e666e3387ba2a4f07ee1f313205e1ac3322137275c38b3...
5,outputs/random_start_full_v14_waveform_recalib...,0.075630,584fc1f8f81231b5a28aab294923a212b0f7b74c3a0cdf...
6,outputs/random_start_full_v14_waveform_recalib...,0.054049,e0899dfbcb2303ceeebb06baf5835175369f68bc5ab5af...
7,outputs/random_start_full_v14_waveform_recalib...,0.018290,561d15d6ebdbe201737ca5457a55b03535aeace5d431df...
8,outputs/random_start_full_v14_waveform_recalib...,0.106974,3e304fb0ddb74f1d0d582f7e2831cc6f621a2503372e54...
9,outputs/random_start_full_v14_waveform_recalib...,0.065802,161c20959a7b77cb6100efb3c4abfb713647dc0db61018...


| Output file                                                                                        | Size, MB | SHA-256 prefix      |
| -------------------------------------------------------------------------------------------------- | -------- | ------------------- |
| outputs/random_start_full_v14_waveform_recalibration/audit_daily_sample.parquet                    | 6.978    | 3cde064a6366a826... |
| outputs/random_start_full_v14_waveform_recalibration/fig1_false_positive_by_window.pdf             | 0.021    | b93c78078e5f9b2e... |
| outputs/random_start_full_v14_waveform_recalibration/fig1_false_positive_by_window.png             | 0.134    | c5dc23f0d5dce7b1... |
| outputs/random_start_full_v14_waveform_recalibration/fig1_false_positive_by_window.svg             | 0.080    | 1e25b445218c01c9... |
| outputs/random_start_full_v14_waveform_recalibration/fig2_pattern_decomposition.pdf                | 0.019    | e666e3387ba2a4f0... |
| outputs/random_start_full_v14_waveform_recalibration/fig2_pattern_decomposition.png                | 0.076    | 584fc1f8f81231b5... |
| outputs/random_start_full_v14_waveform_recalibration/fig2_pattern_decomposition.svg                | 0.054    | e0899dfbcb2303ce... |
| outputs/random_start_full_v14_waveform_recalibration/fig3_study_prevalence_distribution_3month.pdf | 0.018    | 561d15d6ebdbe201... |
| outputs/random_start_full_v14_waveform_recalibration/fig3_study_prevalence_distribution_3month.png | 0.107    | 3e304fb0ddb74f1d... |
| outputs/random_start_full_v14_waveform_recalibration/fig3_study_prevalence_distribution_3month.svg | 0.066    | 161c20959a7b77cb... |
| outputs/random_start_full_v14_waveform_recalibration/fig4_historical_vs_core_definitions.pdf       | 0.019    | cd347af5c667ac4f... |
| outputs/random_start_full_v14_waveform_recalibration/fig4_historical_vs_core_definitions.png       | 0.077    | 4275ccbd2f96c150... |
| outputs/random_start_full_v14_waveform_recalibration/fig4_historical_vs_core_definitions.svg       | 0.051    | 1a144e3ad2552706... |
| outputs/random_start_full_v14_waveform_recalibration/fig4_indeterminate_vs_fpr_frontier.pdf        | 0.027    | 3dba60a484324987... |
| outputs/random_start_full_v14_waveform_recalibration/fig4_indeterminate_vs_fpr_frontier.png        | 0.154    | 2c9d4b55f1ec1762... |
| outputs/random_start_full_v14_waveform_recalibration/fig4_indeterminate_vs_fpr_frontier.svg        | 0.099    | a99440e89f5e87d5... |
| outputs/random_start_full_v14_waveform_recalibration/fig5_qc_null_cycle_day_profile.pdf            | 0.020    | 17ec3170241e60c1... |
| outputs/random_start_full_v14_waveform_recalibration/fig5_qc_null_cycle_day_profile.png            | 0.118    | 3b784eb81505a22e... |
| outputs/random_start_full_v14_waveform_recalibration/fig5_qc_null_cycle_day_profile.svg            | 0.075    | 65bce10a0c34cdd1... |
| outputs/random_start_full_v14_waveform_recalibration/participant_summary.parquet                   | 5.200    | c825fd081426c2d1... |
| outputs/random_start_full_v14_waveform_recalibration/progress.json                                 | 0.003    | bd3a1fe91d1513fb... |
| outputs/random_start_full_v14_waveform_recalibration/study_level_3month.parquet                    | 3.016    | 23e0d1ba06b91ac3... |
| outputs/random_start_full_v14_waveform_recalibration/study_level_3month_n30.parquet                | 3.016    | 23e0d1ba06b91ac3... |
| outputs/random_start_full_v14_waveform_recalibration/summary_tables.csv                            | 4.102    | 8c1946f29e24aaa1... |
| outputs/random_start_full_v14_waveform_recalibration/window_results.parquet                        | 113.562  | 17145b6fa73734de... |

**Table 11 caption.** Machine-readable output provenance. The checksum prefix is included to support reproducibility checks without making the table unnecessarily wide.

## Publication-ready figures

Each figure is written as PNG for notebook viewing and PDF/SVG for publication workflows. Fractional outcomes are displayed on a 0-100% percentage scale. The code cell below is the function call that regenerates the figure set from the populated output tables.

In [14]:
from paper1_null_ce.core.plots import write_all_figures
from tempfile import TemporaryDirectory

# Exercise the exact publication-figure renderer without mutating the immutable
# completed-run bundle or invalidating its recorded checksums.
with TemporaryDirectory(prefix="paper1-notebook-figures-") as figure_dir:
    regenerated = write_all_figures(
        figure_dir,
        summary_tables,
        study_level,
        pd.read_parquet(OUTPUT_DIR / "audit_daily_sample.parquet"),
    )
    regenerated_figure_names = [path.name for path in regenerated]
regenerated_figure_names


['fig1_false_positive_by_window.png',
 'fig1_false_positive_by_window.pdf',
 'fig1_false_positive_by_window.svg',
 'fig2_pattern_decomposition.png',
 'fig2_pattern_decomposition.pdf',
 'fig2_pattern_decomposition.svg',
 'fig3_study_prevalence_distribution_3month.png',
 'fig3_study_prevalence_distribution_3month.pdf',
 'fig3_study_prevalence_distribution_3month.svg',
 'fig4_indeterminate_vs_fpr_frontier.png',
 'fig4_indeterminate_vs_fpr_frontier.pdf',
 'fig4_indeterminate_vs_fpr_frontier.svg',
 'fig4_historical_vs_core_definitions.png',
 'fig4_historical_vs_core_definitions.pdf',
 'fig4_historical_vs_core_definitions.svg',
 'fig5_qc_null_cycle_day_profile.png',
 'fig5_qc_null_cycle_day_profile.pdf',
 'fig5_qc_null_cycle_day_profile.svg']

### Figure 1. False-positive rate by window

Calendar-month false-positive rate for practical monitoring durations, split by cohort.

PDF companion: [PDF version](../outputs/random_start_full_v14_waveform_recalibration/fig1_false_positive_by_window.pdf)

![Figure 1. False-positive rate by window](../outputs/random_start_full_v14_waveform_recalibration/fig1_false_positive_by_window.png)

**Figure caption.** Apparent classification rates are shown as percentages among classifiable participant windows for random calendar windows. The dashed reference line marks 5%.

### Figure 2. C-pattern decomposition

Mutually exclusive C1/C2/C3 pattern categories for full-diary windows.

PDF companion: [PDF version](../outputs/random_start_full_v14_waveform_recalibration/fig2_pattern_decomposition.pdf)

![Figure 2. C-pattern decomposition](../outputs/random_start_full_v14_waveform_recalibration/fig2_pattern_decomposition.png)

**Figure caption.** Bars show the share of all attempted full-diary windows in each pattern category, including indeterminate windows, under strict Herzog phase labeling.

### Figure 3. Study prevalence distribution

Study-level Monte Carlo distribution for 3-month null studies with n=30, n=50, and n=100.

PDF companion: [PDF version](../outputs/random_start_full_v14_waveform_recalibration/fig3_study_prevalence_distribution_3month.pdf)

![Figure 3. Study prevalence distribution](../outputs/random_start_full_v14_waveform_recalibration/fig3_study_prevalence_distribution_3month.png)

**Figure caption.** Each curve summarizes simulated studies using random 3-month windows and the windowed Herzog threshold definition. The y-axis is the proportion of simulated studies; vertical reference lines mark benchmark apparent CE prevalence values.

### Figure 4. Indeterminate versus false-positive frontier

Tradeoff between rejecting underspecified windows and the false-positive rate among classifiable windows.

PDF companion: [PDF version](../outputs/random_start_full_v14_waveform_recalibration/fig4_indeterminate_vs_fpr_frontier.pdf)

![Figure 4. Indeterminate versus false-positive frontier](../outputs/random_start_full_v14_waveform_recalibration/fig4_indeterminate_vs_fpr_frontier.png)

**Figure caption.** Each point is a definition-by-window-by-cohort result. Points farther right have more indeterminate windows; points higher on the plot have more false positives among windows that remained classifiable.

### Appendix Figure. Historical versus core definitions

Assumption-based historical rules compared with core protocol definitions.

PDF companion: [PDF version](../outputs/random_start_full_v14_waveform_recalibration/fig4_historical_vs_core_definitions.pdf)

![Appendix Figure. Historical versus core definitions](../outputs/random_start_full_v14_waveform_recalibration/fig4_historical_vs_core_definitions.png)

**Figure caption.** The historical definitions are exploratory operationalizations and are plotted next to the core definitions only to show their null false-positive behavior under the same 3-month window setting.

### Appendix QC Figure. Null cycle-day seizure profile

Quality-control cycle-day seizure profile in the daily audit sample.

PDF companion: [PDF version](../outputs/random_start_full_v14_waveform_recalibration/fig5_qc_null_cycle_day_profile.pdf)

![Appendix QC Figure. Null cycle-day seizure profile](../outputs/random_start_full_v14_waveform_recalibration/fig5_qc_null_cycle_day_profile.png)

**Figure caption.** The audit sample contains 1% of participant daily rows. Lines show average daily seizure frequency by observed menstrual cycle day with approximate Poisson error bars.

## Interpretation notes

- The notebook is populated from the current files in `outputs/random_start_full_v14_waveform_recalibration`. If those files were produced by smoke mode, the numerical values are smoke-test values, not the final 100,000-participant estimates.
- Full-study values are produced by running `run_paper1_null_ce.py --config config_random_start_full.yaml --full`, then rebuilding this notebook.
- Exact Herzog 2004 results are intentionally present only for 3-complete-cycle windows.
- Historical definitions are assumption-based operationalizations and should be kept separate from core endpoints.
- The manifest assumptions are part of the analysis record:

  - Definition D uses a participant-full-diary method-of-moments negative-binomial alpha recorded in d_alpha; Poisson robust fallback is recorded in d_reason when statsmodels NB fitting fails. Definition D_window_alpha re-estimates alpha from the analyzed window as a non-oracle sensitivity.
  - HORMONE-CYCLE selected diary day 1 uniformly from the first generated cycle.
  - Healthy ovulatory cohort used hormone_cycler build_patient_profile/render_cycle with ovulation_probability set to 1.0 because simulate_diary does not expose a public force-ovulation knob.
  - Historical definitions H1-H4 are assumption-based operationalizations and are flagged in summary outputs.
  - Large-run non-audit participants used the RNG-equivalent compact hormone path; daily hormone concentrations were omitted, while cycle structure and ILP status were retained.
  - Study-level Monte Carlo samples each selected participant from a deterministic pool of precomputed random valid 3-month windows to avoid retaining all daily diaries in memory.
  - The hormone simulator exposes medical-factor knobs but no natural prevalence sampler; heterogeneous menstruating-age medical factors were sampled from config.yaml rates.